# Notebook 03 — Online EWC-style Regularization Baseline
## Class-Incremental Multi-Organ Segmentation — BTCV Dataset

---

### Notebook này làm gì?
| Bước | Nội dung |
|------|----------|
| **Step 1** | Setup: cài thư viện, mount Drive, cấu hình |
| **Step 2** | Copy code từ Notebook 02 (Task defs, Dataset, Model, Loss, Metrics, Trainer) |
| **Step 3** | Compute Fisher Information từ Task 1 checkpoint |
| **Step 4** | EWCTrainer — Trainer với EWC-style penalty |
| **Step 5** | Train Task 2 → 3 → 4 với EWC |
| **Step 6** | Tổng hợp kết quả CIL |

---

### Baseline này là gì?
```
Ý tưởng: Sau Task 1, tính "độ quan trọng" của mỗi weight (Fisher Information).
         Khi train Task 2, phạt nặng nếu thay đổi weights quan trọng.

Loss_total = Loss_segmentation + lambda * SUM( F_i * (theta_i - theta_old_i)^2 )
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                          EWC-style penalty: giữ weights quan trọng gần giá trị cũ
```

---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



---

### Scientific scope statement
This notebook implements an **Online diagonal empirical Fisher EWC** / **Online EWC-style regularization baseline** for class-incremental 2D CT organ segmentation. It does **not** claim exact Fisher estimation or original offline EWC; validation is used only for checkpoint selection, and all paper-reportable final metrics are computed on the held-out TEST split.



## Step 1 — Setup

In [ ]:
!pip install segmentation-models-pytorch torch torchvision nibabel tqdm -q

In [ ]:
import os, json, random, time, copy, gc, csv, csv, csv, csv, csv, csv, csv
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from tqdm import tqdm

# --- Reproducibility ---
# deterministic=True + benchmark=False guarantees bit-exact reproducibility
# across runs on the same hardware as much as PyTorch/cuDNN allow.
# Trade-off: ~5-10% slower than benchmark=True, acceptable for research.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def set_global_seed(seed: int):
    """Set all RNG seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    """Ensure reproducible augmentation across DataLoader workers."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"Deterministic: {torch.backends.cudnn.deterministic} | Benchmark: {torch.backends.cudnn.benchmark}")








In [ ]:
# ============================================================
# REPRODUCIBILITY REPORT
# Print and log all settings that affect result reproducibility.
# This block must appear in paper supplementary or be archivable.
# ============================================================
import platform, sys

print("=" * 65)
print("  REPRODUCIBILITY REPORT")
print("=" * 65)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA version : {torch.version.cuda}")
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    print(f"  cuDNN        : {torch.backends.cudnn.version()}")
print(f"  deterministic: {torch.backends.cudnn.deterministic}  "
      f"(MUST be True for reproducibility)")
print(f"  benchmark    : {torch.backends.cudnn.benchmark}  "
      f"(MUST be False for reproducibility)")
print(f"  Platform     : {platform.platform()}")

# ASSERTION: Fail fast if deterministic was overridden upstream
assert torch.backends.cudnn.deterministic == True, (
    "REPRODUCIBILITY VIOLATION: torch.backends.cudnn.deterministic was set to False. "
    "Check for duplicate import cells that override this setting."
)
assert torch.backends.cudnn.benchmark == False, (
    "REPRODUCIBILITY VIOLATION: torch.backends.cudnn.benchmark was set to True. "
    "This enables non-deterministic cuDNN algorithms."
)
print()
print("  [ASSERTION PASSED] Deterministic settings intact.")
print("=" * 65)







In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

In [ ]:
# ============================================================
# CONFIG
# ============================================================

DRIVE_ROOT     = "/content/drive/MyDrive/Multi-Atlas_Labeling_Beyond_the_Cranial_Vault"
IMAGES_2D_DIR  = f"{DRIVE_ROOT}/data/processed/images"
MASKS_2D_DIR   = f"{DRIVE_ROOT}/data/processed/masks"
METADATA_PATH  = f"{DRIVE_ROOT}/data/processed/metadata.json"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints_ewc_v2"
LOG_DIR        = f"{DRIVE_ROOT}/logs_ewc_v2"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

IGNORE_INDEX = 255

TRAIN_CFG = {
    "batch_size"  : 16,
    "lr"          : 3e-4,
    # 30 epochs minimum for convergence on 512x512 segmentation.
    # 15 epochs shows high epoch-to-epoch variance (±0.15 Dice) = underfitting.
    "n_epochs"    : 30,
    "save_every"  : 10,
    # 3-way split: 70% train / 15% val (checkpoint) / 15% test (final report only)
    # Prevents data leakage: test set is NEVER used for model selection.
    "train_ratio" : 0.70,
    "val_ratio"   : 0.15,
    "test_ratio"  : 0.15,
    "num_workers" : 2,
    "loss_alpha"  : 0.5,
    "encoder"     : "resnet34",
    "pretrained"  : "imagenet",
    "num_classes" : 14,
    "seed"        : 42,
}

# ============================================================
# EWC Hyperparameters — transparent baseline configuration
# ============================================================
#
# ALGORITHM: Online diagonal empirical Fisher EWC.
# This is an Online EWC-style regularization baseline, not exact Fisher EWC
# and not the original offline EWC protocol. old_params is refreshed after
# each task and the diagonal empirical Fisher is accumulated by EMA.
#
# EWC_LAMBDA tuning guide (with per-sample Fisher, normalized penalty):
#   Target: ewc_loss / seg_loss ≈ 0.1–1.0 in first epoch of Task 2.
#   Start at 100, check logs, adjust by 10x increments.
#
EWC_LAMBDA = 100.0

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Multi-seed + small ablation support. Full grid runs are intentionally explicit
# because each run is expensive; summaries below aggregate all completed runs.
EXPERIMENT_SEEDS = [42, 3407, 1234]
ABLATION_GRID = [
    {"name": "finetune", "ewc_lambda": 0.0,   "use_ewc": False},
    {"name": "ewc_lam100", "ewc_lambda": 100.0, "use_ewc": True},
    {"name": "ewc_lam500", "ewc_lambda": 500.0, "use_ewc": True},
]
CURRENT_SEED = TRAIN_CFG["seed"]
CURRENT_ABLATION = {"name": "ewc_lam100", "ewc_lambda": EWC_LAMBDA, "use_ewc": True}
RUN_FULL_EXPERIMENT_GRID = False  # set True only when intentionally launching all 9 runs

# Fisher computed with batch_size=1 (per-sample gradients, no batch-averaging
# bias). Slower but mathematically correct. With batch>1, the squared mean
# gradient underestimates the mean squared gradient (Cauchy-Schwarz).
FISHER_BATCH_SIZE = 1

# Number of samples for Fisher estimation. 200 samples × 512×512 provides
# good coverage for diagonal FIM approximation.
FISHER_SAMPLES = 200

# Online EWC exponential moving average decay for Fisher accumulation.
# F_new = DECAY * F_old + (1 - DECAY) * F_task
# decay=0.9 gives ~50% weight to last 7 tasks (geometric series).
EWC_FISHER_DECAY = 0.9

# CIL evaluation frequency (epochs). Run every 5 epochs for 30-epoch training.
CIL_EVAL_EVERY = 5

# ============================================================
set_global_seed(TRAIN_CFG["seed"])

print("Config (Online diagonal empirical Fisher EWC baseline):")
print(f"   Algorithm       : Online diagonal empirical Fisher EWC-style baseline")
print(f"   EWC lambda      : {EWC_LAMBDA}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Seeds           : {EXPERIMENT_SEEDS}")
print(f"   Ablations       : {[a['name'] for a in ABLATION_GRID]}")
print(f"   Fisher batch    : {FISHER_BATCH_SIZE} (per-sample, unbiased)")
print(f"   Fisher samples  : {FISHER_SAMPLES}")
print(f"   Fisher decay    : {EWC_FISHER_DECAY}")
print(f"   CIL eval every  : {CIL_EVAL_EVERY} epochs")
print(f"   Data split      : {TRAIN_CFG['train_ratio']}/{TRAIN_CFG['val_ratio']}/{TRAIN_CFG['test_ratio']} (train/val/test)")
for k, v in TRAIN_CFG.items():
    if k not in ("train_ratio", "val_ratio", "test_ratio"):
        print(f"   {k:15s}: {v}")


def configure_experiment_run(seed, ablation):
    """Configure one seed/ablation run without changing the train/val/test split."""
    global CURRENT_SEED, CURRENT_ABLATION, EWC_LAMBDA
    global CHECKPOINT_DIR, LOG_DIR, USE_EWC_REGULARIZATION

    CURRENT_SEED = int(seed)
    CURRENT_ABLATION = dict(ablation)
    EWC_LAMBDA = float(CURRENT_ABLATION["ewc_lambda"])
    USE_EWC_REGULARIZATION = bool(CURRENT_ABLATION["use_ewc"])
    CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints_ewc_v2/{CURRENT_ABLATION['name']}/seed_{CURRENT_SEED}"
    LOG_DIR = f"{DRIVE_ROOT}/logs_ewc_v2/{CURRENT_ABLATION['name']}/seed_{CURRENT_SEED}"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)
    TRAIN_CFG["seed"] = CURRENT_SEED
    set_global_seed(CURRENT_SEED)
    print(f"[RUN] seed={CURRENT_SEED} ablation={CURRENT_ABLATION['name']} "
          f"lambda={EWC_LAMBDA} use_ewc={USE_EWC_REGULARIZATION}")
    print(f"[RUN] checkpoints={CHECKPOINT_DIR}")
    print(f"[RUN] logs={LOG_DIR}")

USE_EWC_REGULARIZATION = bool(CURRENT_ABLATION["use_ewc"])








## Step 2 — Code tu Notebook 02 (Task defs, Dataset, Model, Loss, Metrics, Trainer)

In [ ]:
# ============================================================
# Task Definitions (giong het Notebook 02)
# ============================================================
TASK_ORGANS = {
    1: [6, 7],           # Liver(6), Stomach(7)             — cơ quan LỚN nhất
    2: [1, 2, 3, 8],     # R.Kidney(2), L.Kidney(3),
                         # Spleen(1), Aorta(8)             — cơ quan TRUNG BÌNH
    3: [4, 9, 10, 11],   # Gallbladder(4), IVC(9),
                         # Portal Vein(10), Pancreas(11)    — cơ quan NHỎ
    4: [5, 12, 13],      # Esophagus(5), R.Adrenal(12),
                         # L.Adrenal(13)                    — cơ quan RẤT NHỎ
}
ORGAN_NAMES = {
    0:  "Background",
    1:  "Spleen (Lach)",
    2:  "Right Kidney (Than P)",
    3:  "Left Kidney (Than T)",
    4:  "Gallbladder (Tui mat)",
    5:  "Esophagus (Thuc quan)",
    6:  "Liver (Gan)",
    7:  "Stomach (Da day)",
    8:  "Aorta (DM chu)",
    9:  "IVC (TM chu duoi)",
    10: "Portal Vein (TM cua)",
    11: "Pancreas (Tuy)",
    12: "Right Adrenal (TTT P)",
    13: "Left Adrenal (TTT T)",
}

ALL_PAST_ORGANS = {
    1: [6, 7],
    2: [6, 7, 1, 2, 3, 8],
    3: [6, 7, 1, 2, 3, 8, 4, 9, 10, 11],
    4: [6, 7, 1, 2, 3, 8, 4, 9, 10, 11, 5, 12, 13],
}

print("Task definitions:")
for task_id, organs in TASK_ORGANS.items():
    print(f"   Task {task_id}: {[ORGAN_NAMES[o] for o in organs]}")

In [ ]:
# ============================================================
# Helper Functions (giong het Notebook 02)
# ============================================================

def remap_mask_for_task(mask_npy, task_id, ignore_index=255):
    task_organs = TASK_ORGANS[task_id]
    new_mask = np.full_like(mask_npy, fill_value=ignore_index, dtype=np.uint8)
    new_mask[mask_npy == 0] = 0
    for organ_id in task_organs:
        new_mask[mask_npy == organ_id] = organ_id
    return new_mask


def get_slices_for_task(metadata, task_id):
    task_organs = set(TASK_ORGANS[task_id])
    return [r for r in metadata if task_organs & set(r["organs_present"])]


def log_vram(label=""):
    if not torch.cuda.is_available():
        return
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    pct   = used / total * 100
    tag   = "WARNING" if pct > 85 else "OK"
    print(f"   {tag} VRAM {label}: {used:.2f}/{total:.1f} GB ({pct:.0f}%)")

print("Helpers ready!")

In [ ]:
# ============================================================
# BTCVDataset (giong het Notebook 02)
# ============================================================
class BTCVDataset(Dataset):
    def __init__(self, records, images_dir, masks_dir,
                 task_id=None, augment=False, ignore_index=255):
        self.records      = records
        self.images_dir   = images_dir
        self.masks_dir    = masks_dir
        self.task_id      = task_id
        self.augment      = augment
        self.ignore_index = ignore_index

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        ct_npy   = np.load(os.path.join(self.images_dir, rec["filename"]))
        mask_npy = np.load(os.path.join(self.masks_dir,  rec["filename"]))

        if self.task_id is not None:
            mask_npy = remap_mask_for_task(mask_npy, self.task_id, self.ignore_index)

        if self.augment:
            ct_npy, mask_npy = self._augment(ct_npy, mask_npy)

        ct_tensor   = torch.from_numpy(ct_npy.copy()).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask_npy.astype(np.int64))
        return ct_tensor, mask_tensor

    def _augment(self, ct, mask):
        if np.random.random() > 0.5:
            ct   = np.fliplr(ct).copy()
            mask = np.fliplr(mask).copy()
        if np.random.random() > 0.5:
            ct   = np.flipud(ct).copy()
            mask = np.flipud(mask).copy()
        k = np.random.randint(0, 4)
        if k > 0:
            ct   = np.rot90(ct,   k).copy()
            mask = np.rot90(mask, k).copy()
        return ct, mask

print("BTCVDataset ready!")

In [ ]:
# ============================================================
# Model + Loss + Metrics (giong het Notebook 02)
# ============================================================

def build_unet(num_classes=14, encoder="resnet34", pretrained="imagenet"):
    return smp.Unet(
        encoder_name    = encoder,
        encoder_weights = pretrained,
        in_channels     = 1,
        classes         = num_classes,
        activation      = None,
    )


class TaskDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5, ignore_index=255):
        super().__init__()
        self.smooth       = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target, task_organs):
        probs = F.softmax(logits, dim=1)
        valid = (target != self.ignore_index).float()
        dice_list = []
        # CHI tinh foreground organs, KHONG tinh background (class 0)
        # Ly do: Dice_bg luon ~1.0 (bg chiem >90% pixels)
        #        -> keo loss xuong thap gia tao -> model khong hoc tot organs nho
        for c in task_organs:
            pred_c = probs[:, c] * valid
            true_c = (target == c).float() * valid
            intsec = (pred_c * true_c).sum(dim=(1, 2))
            union  = pred_c.sum(dim=(1, 2)) + true_c.sum(dim=(1, 2))
            dice   = (2 * intsec + self.smooth) / (union + self.smooth)
            dice_list.append(1 - dice.mean())
        return torch.stack(dice_list).mean()


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5, ignore_index=255):
        super().__init__()
        self.alpha        = alpha
        self.ignore_index = ignore_index
        self.ce           = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.dice         = TaskDiceLoss(ignore_index=ignore_index)

    def forward(self, logits, target, task_organs):
        loss_ce   = self.ce(logits, target)
        loss_dice = self.dice(logits, target, task_organs)
        total     = self.alpha * loss_ce + (1 - self.alpha) * loss_dice
        return total, loss_ce.item(), loss_dice.item()


def format_dice_table(dice_dict, task_organs, epoch=None):
    header = f"  {'Epoch':>6}" if epoch is not None else ""
    print(f"\n{header}  {'Organ':30s}  {'Dice':>8}  {'Status':>10}")
    print("  " + "-" * 58)
    valid_dices = []
    for organ_id in task_organs:
        dice = dice_dict.get(organ_id, float('nan'))
        name = ORGAN_NAMES.get(organ_id, f"Organ {organ_id}")
        if not np.isnan(dice):
            valid_dices.append(dice)
            status = "Good" if dice >= 0.7 else ("Fair" if dice >= 0.5 else "Poor")
            ep_str = f"{epoch:6d}" if epoch is not None else ""
            print(f"  {ep_str}  {name:30s}  {dice:8.4f}  {status}")
        else:
            ep_str = f"{epoch:6d}" if epoch is not None else ""
            print(f"  {ep_str}  {name:30s}  {'N/A':>8}  (not in batch)")
    if valid_dices:
        mean_dice = np.mean(valid_dices)
        print("  " + "=" * 58)
        ep_str = f"{epoch:6d}" if epoch is not None else ""
        print(f"  {ep_str}  {'Mean Dice':30s}  {mean_dice:8.4f}")
    return np.mean(valid_dices) if valid_dices else 0.0


# ============================================================
# HAM MOI: compute_iou_per_organ
# Tinh global IoU (tich luy intersection/union tren toan batch)
# cho danh sach organ_ids cho truoc.
#
# Cong thuc:
#   IoU  = intersection / union
#   DSC  = 2*inter / (pred_sum + true_sum)
#   Moi quan he: IoU = DSC / (2 - DSC)
#
# KHONG tinh per-sample roi lay mean (vi organs nho co rat it pixels,
# de bi bien dong). Thay vao do tich luy inter/union tren TOAN BO batch
# (global IoU) de co ket qua on dinh hon.
# ============================================================
def compute_iou_per_organ(pred, target, organ_ids, ignore_index=255):
    """
    Tinh global IoU cho tung organ bang cach tich luy inter/union.

    Args:
        pred       : torch.Tensor (B, H, W) - predicted class indices
        target     : torch.Tensor (B, H, W) - ground truth, ignore_index=255
        organ_ids  : list[int] - danh sach organ channel can tinh
        ignore_index: int - nhan bi bo qua (mac dinh 255)

    Returns:
        dict {organ_id: iou_score}
        - float('nan') neu organ khong xuat hien (union=0)
        - Dung float('nan') thay 0 de tranh lam sai mIoU
    """
    valid_mask   = (target != ignore_index)
    pred_valid   = pred[valid_mask]
    target_valid = target[valid_mask]

    iou_dict = {}
    for organ_id in organ_ids:
        pred_o   = (pred_valid   == organ_id)
        target_o = (target_valid == organ_id)

        intersection = (pred_o & target_o).sum().item()
        union        = (pred_o | target_o).sum().item()

        if union == 0:
            iou_dict[organ_id] = float('nan')
        else:
            iou_dict[organ_id] = intersection / union

    return iou_dict


print("Model + Loss + Metrics + compute_iou_per_organ ready!")


## Step 3 — Compute Fisher Information tu Task 1

In [ ]:
# ============================================================
# EWCRegularizer — Online diagonal empirical Fisher module
# ============================================================
#
# ALGORITHM: Online diagonal empirical Fisher EWC-style regularization baseline
#
# KEY FIXES over previous version:
#   FIX-1  Fisher uses reduction='none' + spatial sum → per-sample gradient,
#          divided by valid pixel count.  Eliminates "pixel-averaging scale
#          catastrophe" where Fisher ~ 1/(H*W)^2 making penalty inactive.
#   FIX-2  Task-specific Fisher: loss computed only on learned-class mask
#          (not full 14-class mask), so Fisher reflects actual learned knowledge.
#   FIX-3  Channel-wise Fisher masking: Fisher zeroed for classifier channels
#          corresponding to classes NOT yet learned (avoids penalising random
#          initialisations of future-class weights).
#   FIX-4  Background Fisher zeroed in final conv: background semantics shift
#          every task, so EWC must not freeze background classifier weights.
#   FIX-5  Penalty normalised by num_params so lambda is scale-invariant
#          regardless of model size.
#   FIX-6  register_buffer for Fisher & old_params → proper device management,
#          no manual .to(device) calls.
#   FIX-7  Vectorised penalty (no Python for-loop in forward pass).
#   FIX-8  FISHER_BATCH_SIZE=1 → unbiased per-sample Fisher.
#   FIX-9  .detach().clone().requires_grad_(False) instead of .data.clone().
#
# ============================================================

class EWCRegularizer(nn.Module):
    """
    Modular Online diagonal empirical Fisher EWC-style regularizer for class-incremental segmentation.

    Usage:
        ewc = EWCRegularizer(model, num_classes=14, device=DEVICE)
        ewc.compute_and_set_fisher(model, loader, task_organs, num_samples=200)
        loss_ewc = ewc(model)  # call in training loop
    """

    # Name of the final segmentation conv layer in SMP U-Net
    # (smp.Unet output block is model.segmentation_head[0])
    FINAL_CONV_NAME = "segmentation_head.0"

    def __init__(self, model: nn.Module, num_classes: int, device: torch.device,
                 ewc_lambda: float = 100.0, decay: float = 0.9):
        super().__init__()
        self.num_classes  = num_classes
        self.device       = device
        self.ewc_lambda   = ewc_lambda
        self.decay        = decay
        self._param_names = []  # ordered list of learnable param names
        self._num_params  = 0

        # Collect learnable parameter metadata
        for name, param in model.named_parameters():
            if param.requires_grad:
                self._param_names.append(name)
                self._num_params += param.numel()

        # Register Fisher and old_params as buffers so they follow .to(device)
        # automatically and are excluded from optimizer updates.
        for name in self._param_names:
            safe = name.replace(".", "__")
            self.register_buffer(f"fisher_{safe}",
                                 torch.zeros(1))   # placeholder; resized on first update
            self.register_buffer(f"optpar_{safe}",
                                 torch.zeros(1))

        self._fisher_initialised = False
        self._learned_classes    = []  # grows as tasks are completed

        print(f"EWCRegularizer: {len(self._param_names)} param tensors, "
              f"{self._num_params:,} total params")

    # ----------------------------------------------------------
    def _get_fisher(self, name: str) -> torch.Tensor:
        return getattr(self, f"fisher_{name.replace('.', '__')}")

    def _get_optpar(self, name: str) -> torch.Tensor:
        return getattr(self, f"optpar_{name.replace('.', '__')}")

    def _set_fisher(self, name: str, val: torch.Tensor):
        setattr(self, f"fisher_{name.replace('.', '__')}", val)

    def _set_optpar(self, name: str, val: torch.Tensor):
        setattr(self, f"optpar_{name.replace('.', '__')}", val)

    # ----------------------------------------------------------
    @torch.no_grad()
    def _build_channel_mask(self, model: nn.Module) -> dict:
        """
        FIX-3 + FIX-4: Build per-parameter channel masks.

        For the final conv weight (shape [num_classes, C_in, kH, kW]):
          - Channels in self._learned_classes → mask = 1.0
          - Channel 0 (background)            → mask = 0.0  (FIX-4)
          - Unlearned future channels          → mask = 0.0  (FIX-3)

        For all other parameters → mask = 1.0 (full Fisher applied).

        Returns dict: {param_name: mask_tensor (same shape as param)}
        """
        masks = {}
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if self.FINAL_CONV_NAME in name and "weight" in name:
                # param shape: (num_classes, C_in, kH, kW)
                mask = torch.zeros_like(param)
                for c in self._learned_classes:
                    if c != 0 and c < self.num_classes:  # skip background
                        mask[c] = 1.0
                masks[name] = mask
            elif self.FINAL_CONV_NAME in name and "bias" in name:
                # bias shape: (num_classes,)
                mask = torch.zeros_like(param)
                for c in self._learned_classes:
                    if c != 0 and c < self.num_classes:
                        mask[c] = 1.0
                masks[name] = mask
            else:
                masks[name] = torch.ones_like(param)
        return masks

    # ----------------------------------------------------------
    def compute_and_set_fisher(self, model: nn.Module, dataloader,
                               task_organs: list, num_samples: int = 200):
        """
        FIX-1 + FIX-2: Compute empirical diagonal Fisher with:
          - Task-specific loss (only learned classes in task_organs).
          - reduction='none' + spatial sum → per-sample gradient.
          - Divide squared gradient by valid pixel count (not image count).
          - FISHER_BATCH_SIZE=1 recommended (unbiased per-sample estimate).

        Accumulate into existing Fisher via Online diagonal empirical Fisher EWC EMA (FIX-6 via buffers).

        Args:
            model       : model at optimal params for the completed task
            dataloader  : task-specific DataLoader (task_id set, augment=False)
                          MUST use FISHER_BATCH_SIZE=1 for unbiased Fisher
            task_organs : list of class indices learned in this task (e.g. [6,7])
            num_samples : max number of samples to process
        """
        model.eval()

        # Accumulators: sum of per-sample squared gradients and pixel counts
        fisher_new  = {n: torch.zeros_like(p.detach())
                       for n, p in model.named_parameters() if p.requires_grad}
        total_pixels = 0  # total valid pixels seen (for normalisation)
        count        = 0

        log_vram("pre-Fisher")

        for ct_batch, mask_batch in dataloader:
            if count >= num_samples:
                break

            ct_batch   = ct_batch.to(self.device)   # (1, 1, H, W) with batch=1
            mask_batch = mask_batch.to(self.device)  # (1, H, W)

            if count == 0:
                print(f"   [Fisher] ct: {tuple(ct_batch.shape)}  "
                      f"mask unique: {mask_batch.unique().tolist()[:8]}")

            # Count valid pixels in this sample
            valid_pixels = (mask_batch != IGNORE_INDEX).sum().item()
            if valid_pixels == 0:
                count += ct_batch.size(0)
                continue

            model.zero_grad()
            logits = model(ct_batch)  # (1, num_classes, H, W)

            # FIX-1: reduction='none' → per-pixel loss tensor (1, H, W)
            # FIX-2: task-specific mask (only task_organs classes labeled)
            loss_map = F.cross_entropy(
                logits, mask_batch,
                ignore_index=IGNORE_INDEX,
                reduction='none'   # shape: (B, H, W)
            )
            # Sum spatially (not mean) to get total log-likelihood contribution
            # per sample, then backward. This gives gradients proportional to
            # the total loss, not the pixel-averaged loss.
            loss_sum = loss_map.sum()
            loss_sum.backward()

            for n, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    # Per-sample squared gradient accumulated
                    fisher_new[n] += p.grad.detach().pow(2)

            total_pixels += valid_pixels
            count += ct_batch.size(0)

        # Normalise by total valid pixel count (not sample count, not image count)
        # This makes Fisher scale-invariant to image resolution and dataset size.
        denom = max(total_pixels, 1)
        for n in fisher_new:
            fisher_new[n] /= denom

        # Apply channel mask (FIX-3 + FIX-4)
        channel_masks = self._build_channel_mask(model)
        for n in fisher_new:
            if n in channel_masks:
                fisher_new[n] *= channel_masks[n].to(self.device)

        # Validate Fisher
        mean_f = float(np.mean([f.mean().item() for f in fisher_new.values()]))
        max_f  = float(np.max([f.max().item() for f in fisher_new.values()]))
        assert all(f.min().item() >= 0 for f in fisher_new.values()), \
            "Negative Fisher values detected — check loss/gradient sign"
        assert not any(f.isnan().any() for f in fisher_new.values()), \
            "NaN in Fisher — check for numerical instability"
        print(f"   Fisher (task-specific, pixel-normalised): "
              f"{count} samples, {total_pixels} valid pixels, "
              f"mean={mean_f:.4e}, max={max_f:.4e}")

        # Online diagonal empirical Fisher EWC accumulation (FIX-6): EMA of Fisher across tasks
        if not self._fisher_initialised:
            for n in fisher_new:
                self._set_fisher(n, fisher_new[n].to(self.device))
            self._fisher_initialised = True
            print(f"   Fisher initialised (first task).")
        else:
            for n in fisher_new:
                f_old = self._get_fisher(n)
                f_merged = self.decay * f_old + (1.0 - self.decay) * fisher_new[n]
                self._set_fisher(n, f_merged)
            print(f"   Fisher EMA updated (decay={self.decay}).")

        # Store current model params as reference (old_params)
        # FIX-9: use .detach().clone().requires_grad_(False)
        for n, p in model.named_parameters():
            if p.requires_grad:
                self._set_optpar(n, p.detach().clone().requires_grad_(False))

        log_vram("post-Fisher")

    # ----------------------------------------------------------
    def forward(self, model: nn.Module) -> torch.Tensor:
        """
        Compute EWC penalty (vectorised, no Python loop in grad path).

        FIX-5: Normalise by num_params so lambda is model-size-invariant.
        L_EWC = lambda * (1/N_params) * sum_i F_i * (theta_i - theta*_i)^2

        Returns scalar tensor (on self.device) with grad_fn attached.
        """
        if not self._fisher_initialised:
            return torch.tensor(0.0, device=self.device, requires_grad=False)

        penalty_terms = []
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            f   = self._get_fisher(n)    # detached buffer
            ref = self._get_optpar(n)    # detached buffer
            diff = p - ref               # has grad_fn (through p)
            penalty_terms.append((f * diff.pow(2)).sum())

        if not penalty_terms:
            return torch.tensor(0.0, device=self.device, requires_grad=False)

        raw_penalty = torch.stack(penalty_terms).sum()
        # FIX-5: normalise by total param count for lambda scale-invariance
        return self.ewc_lambda * raw_penalty / self._num_params

    # ----------------------------------------------------------
    def update_learned_classes(self, task_organs: list):
        """Track which classes have been learned (for channel masking)."""
        for c in task_organs:
            if c not in self._learned_classes:
                self._learned_classes.append(c)
        print(f"   Learned classes: {sorted(self._learned_classes)}")

    # ----------------------------------------------------------

    def fisher_diagnostics(self):
        """Return fail-fast diagnostics for the diagonal empirical Fisher state."""
        vals = []
        for name in self._param_names:
            f = getattr(self, self._buf_name(name, "fisher"))
            vals.append(f.detach().float().reshape(-1))
        if not vals:
            raise RuntimeError("EWC diagnostics failed: Fisher state is empty")
        flat = torch.cat(vals)
        if not torch.isfinite(flat).all():
            raise FloatingPointError("NaN/Inf Fisher detected")
        stats = {
            "fisher_min": float(flat.min().item()),
            "fisher_mean": float(flat.mean().item()),
            "fisher_max": float(flat.max().item()),
            "fisher_sum": float(flat.sum().item()),
        }
        stats["fisher_collapse"] = stats["fisher_mean"] < 1e-12 or stats["fisher_sum"] <= 0.0
        stats["fisher_explosion"] = stats["fisher_max"] > 1e6 or stats["fisher_mean"] > 1e3
        return stats

    def parameter_drift_norms(self, model: nn.Module):
        """Return L2/Linf drift from the stored online EWC reference parameters."""
        sq_sum = 0.0
        linf = 0.0
        n_params = 0
        named = dict(model.named_parameters())
        for name in self._param_names:
            if name not in named:
                raise KeyError(f"Missing parameter during drift diagnostics: {name}")
            old = getattr(self, self._buf_name(name, "optpar"))
            diff = (named[name].detach().float() - old.detach().float()).reshape(-1)
            sq_sum += float(torch.sum(diff * diff).item())
            linf = max(linf, float(torch.max(torch.abs(diff)).item())) if diff.numel() else linf
            n_params += diff.numel()
        return {"drift_l2": float(np.sqrt(sq_sum)), "drift_linf": linf, "drift_params": int(n_params)}

    def save(self, path: str, task_id: int):
        """Save full EWC state to disk (Colab-crash resilient)."""
        state = {
            "fisher":           {n: self._get_fisher(n).cpu()
                                 for n in self._param_names},
            "optpar":           {n: self._get_optpar(n).cpu()
                                 for n in self._param_names},
            "learned_classes":  self._learned_classes,
            "task_id":          task_id,
            "ewc_lambda":       self.ewc_lambda,
            "decay":            self.decay,
            "fisher_initialised": self._fisher_initialised,
        }
        torch.save(state, path)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"EWC state saved → {path} ({size_mb:.1f} MB)")

    @classmethod
    def load(cls, path: str, model: nn.Module, num_classes: int,
             device: torch.device) -> "EWCRegularizer":
        """Restore EWC state from disk."""
        state = torch.load(path, map_location=device, weights_only=False)
        ewc = cls(model, num_classes=num_classes, device=device,
                  ewc_lambda=state["ewc_lambda"], decay=state["decay"])
        for n in state["fisher"]:
            ewc._set_fisher(n, state["fisher"][n].to(device))
            ewc._set_optpar(n, state["optpar"][n].to(device))
        ewc._learned_classes    = state["learned_classes"]
        ewc._fisher_initialised = state["fisher_initialised"]
        print(f"EWC state loaded ← {path} (after task {state['task_id']}, "
              f"learned={sorted(ewc._learned_classes)})")
        return ewc


# ============================================================
# Helpers: safe JSON serialisation + mIoU history I/O
# ============================================================

def _safe_float(v):
    """Convert numpy scalars / None to plain Python float for JSON."""
    if v is None:
        return None
    try:
        f = float(v)
        return None if (f != f) else round(f, 6)  # NaN check
    except (TypeError, ValueError):
        return None

def save_miou_history(history: list, path: str):
    with open(path, 'w') as f:
        json.dump({"miou_history": [float(x) for x in history]}, f, indent=2)
    print(f"   mIoU history → {path}: {[round(float(x),4) for x in history]}")

def load_miou_history(path: str) -> list:
    if not os.path.exists(path):
        print(f"   WARNING: mIoU history not found: {path}")
        return []
    with open(path) as f:
        return [float(x) for x in json.load(f)["miou_history"]]

def save_organ_baselines(baselines: dict, path: str):
    with open(path, 'w') as f:
        json.dump({str(k): float(v) for k, v in baselines.items()}, f, indent=2)
    print(f"   Organ baselines → {path} ({len(baselines)} organs)")

def load_organ_baselines(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {int(k): float(v) for k, v in json.load(f).items()}

def update_organ_baselines(existing: dict, new_task_dice: dict,
                           new_task_organs: list, path: str) -> dict:
    """Only write baseline for NEW organs (never overwrite earlier baselines)."""
    for oid in new_task_organs:
        if oid not in existing:
            val = new_task_dice.get(oid, float('nan'))
            if not (val != val):  # not NaN
                existing[oid] = float(val)
    save_organ_baselines(existing, path)
    return existing


print("EWCRegularizer (online diagonal empirical Fisher baseline) ready!")
print("  FIX-1: reduction='none' + spatial sum → unbiased Fisher")
print("  FIX-2: task-specific mask (not full 14-class mask)")
print("  FIX-3: channel mask — future classes not penalised")
print("  FIX-4: background Fisher zeroed in final conv")
print("  FIX-5: penalty normalised by num_params")
print("  FIX-6: register_buffer for Fisher/optpar")
print("  FIX-7: vectorised penalty (torch.stack)")
print("  FIX-8: FISHER_BATCH_SIZE=1 (per-sample)")
print("  FIX-9: .detach().clone().requires_grad_(False)")









## Step 4 — EWCTrainer

In [ ]:
# ============================================================
# EWCTrainer — trainer with online EWC-style regularization
# ============================================================
# Changes from v1:
#   - self.ewc (EWCRegularizer) replaces self.fisher / self.old_params dicts
#   - AMP: torch.amp.autocast / torch.amp.GradScaler (not deprecated cuda.amp)
#   - EWC penalty computed via ewc(model) — vectorised, normalised
#   - _safe_round → _safe_float (avoids np.float64 leak into JSON)
#   - ewc_ratio logged each epoch (ewc_loss/seg_loss) to confirm EWC is active
#   - evaluate_cil accepts an explicit split label; final reports must pass TEST loader
#   - save_checkpoint no longer serialises fisher/optpar (managed by EWCRegularizer)
# ============================================================

class EWCTrainer:
    def __init__(self, model, optimizer, criterion, scheduler,
                 device, task_id, checkpoint_dir, log_dir,
                 ewc_regularizer=None,
                 save_every=10, ignore_index=255, cil_eval_every=5):
        self.model          = model
        self.optimizer      = optimizer
        self.criterion      = criterion
        self.scheduler      = scheduler
        self.device         = device
        self.task_id        = task_id
        self.task_organs    = TASK_ORGANS[task_id]
        self.all_learned    = ALL_PAST_ORGANS[task_id]
        self.checkpoint_dir = checkpoint_dir
        self.log_dir        = log_dir
        self.save_every     = save_every
        self.ignore_index   = ignore_index
        self.cil_eval_every = cil_eval_every

        # EWCRegularizer object (None for Task 1, set for Task 2+)
        self.ewc = ewc_regularizer

        self.best_val_dice  = 0.0
        self.best_cil_dice  = 0.0
        self.history        = {"train_loss": [], "val_loss": [],
                                "val_dice":   [], "lr":       [],
                                "ewc_loss":   [], "ewc_ratio":[], "cil_dice": []}
        self.miou_history   = []

        # AMP: use non-deprecated torch.amp API (PyTorch ≥ 2.0)
        self.use_amp = (device.type == 'cuda')
        self.scaler  = torch.amp.GradScaler('cuda', enabled=self.use_amp)
        ewc_status   = (f"ACTIVE (λ={self.ewc.ewc_lambda})"
                        if self.ewc is not None else "OFF (Task 1)")
        print(f"  AMP: {self.use_amp}  |  EWC: {ewc_status}")

    # ----------------------------------------------------------
    def train_epoch(self, loader):
        """Train one epoch with EWC penalty + AMP (non-deprecated API)."""
        self.model.train()
        total_loss = total_ce = total_dice = total_ewc = 0.0
        first_batch = not hasattr(self, '_shape_logged')

        for ct_batch, mask_batch in loader:
            ct_batch   = ct_batch.to(self.device, non_blocking=True)
            mask_batch = mask_batch.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            # Segmentation loss under AMP autocast (FP16 forward pass)
            with torch.amp.autocast('cuda', enabled=self.use_amp):
                logits = self.model(ct_batch)

                if first_batch:
                    print(f"   [TRAIN] ct={tuple(ct_batch.shape)}, "
                          f"mask={tuple(mask_batch.shape)}, "
                          f"logits={tuple(logits.shape)}")
                    self._shape_logged = True
                    first_batch = False

                loss_seg, ce_v, dice_v = self.criterion(
                    logits, mask_batch, self.task_organs
                )

            # EWC penalty computed in FP32 outside autocast.
            # EWCRegularizer.forward() uses vectorised torch.stack — no loop.
            # Penalty is already normalised by num_params inside EWCRegularizer.
            loss_ewc = (self.ewc(self.model)
                        if self.ewc is not None else
                        torch.tensor(0.0, device=self.device))

            loss = loss_seg.float() + loss_ewc

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()
            total_ce   += ce_v
            total_dice += dice_v
            total_ewc  += loss_ewc.item()

        n = len(loader)
        return total_loss/n, total_ce/n, total_dice/n, total_ewc/n

    # ----------------------------------------------------------
    @torch.no_grad()
    def validate(self, loader):
        """Validate chi tren task_organs hien tai (dung cho scheduler + log)."""
        self.model.eval()
        total_loss = 0.0
        inter = defaultdict(float)
        union = defaultdict(float)

        for ct_batch, mask_batch in loader:
            ct_batch   = ct_batch.to(self.device, non_blocking=True)
            mask_batch = mask_batch.to(self.device, non_blocking=True)
            logits = self.model(ct_batch)
            loss, _, _ = self.criterion(logits, mask_batch, self.task_organs)
            total_loss += loss.item()

            pred  = torch.argmax(logits, dim=1)
            valid = (mask_batch != self.ignore_index)
            for organ_id in self.task_organs:
                pred_o = (pred  == organ_id) & valid
                true_o = (mask_batch == organ_id) & valid
                inter[organ_id] += (pred_o & true_o).sum().item()
                union[organ_id] += pred_o.sum().item() + true_o.sum().item()

        val_loss = total_loss / len(loader)
        dice_dict = {}
        for organ_id in self.task_organs:
            if union[organ_id] > 0:
                dice_dict[organ_id] = 2 * inter[organ_id] / union[organ_id]
            else:
                dice_dict[organ_id] = float('nan')
        return val_loss, dice_dict

    # ----------------------------------------------------------
    @torch.no_grad()
    def evaluate_cil(self, test_loader, split="test"):
        """
        Evaluate tren TAT CA organs da hoc sau task hien tai.

        Thay doi so voi ban cu:
          - Tich luy ca inter_dice/union_dice (cho DSC)
            lan inter_iou/union_iou (cho IoU) trong CUNG 1 vong lap
            -> tinh kiem de tranh duyet lai loader 2 lan
          - Tra ve them: iou_all, iou_old, iou_new
          - Giu nguyen logic global (khong per-sample)
          - test_loader must use task_id=None (full mask 0-13, no remap)

        Returns:
            dice_all, dice_old, dice_new  : dict {organ_id: dsc_score}
            iou_all,  iou_old,  iou_new   : dict {organ_id: iou_score}
        """
        self.model.eval()

        # Tich luy intersection va union (global, khong per-sample)
        # inter_d / union_d -> de tinh DSC  (DSC = 2*inter / union)
        # inter_i / union_i -> de tinh IoU  (IoU = inter / union)
        # Luu y: union_d = pred_sum + true_sum  (khac voi union IoU)
        #         union_i = |pred OR true|        (tinh bang bit operation)
        inter_d = defaultdict(float)
        union_d = defaultdict(float)
        inter_i = defaultdict(float)
        union_i = defaultdict(float)

        first_batch = True  # IMP-01: in shape lan dau

        assert split in {"val", "test"}, f"Invalid evaluation split: {split}"
        if split == "test":
            print("  [TEST] Final held-out evaluation; validation data is not used.")
        else:
            print("  [VAL] CIL checkpoint-selection evaluation only; not paper-reportable.")

        for ct_batch, mask_batch in test_loader:
            ct_batch   = ct_batch.to(self.device, non_blocking=True)
            mask_batch = mask_batch.to(self.device, non_blocking=True)
            logits = self.model(ct_batch)
            pred   = torch.argmax(logits, dim=1)  # (B, H, W)

            # IMP-01: Kiem tra shape tensor lan dau tien
            if first_batch:
                print(f"   [eval_cil] ct_batch  : {tuple(ct_batch.shape)}")
                print(f"   [eval_cil] mask_batch: {tuple(mask_batch.shape)}, "
                      f"unique vals: {mask_batch.unique().tolist()[:10]}")
                print(f"   [eval_cil] logits    : {tuple(logits.shape)}")
                print(f"   [eval_cil] pred      : {tuple(pred.shape)}")
                first_batch = False

            # Loai bo pixels ignore_index=255 (nhat quan voi validate)
            valid = (mask_batch != self.ignore_index)  # (B, H, W) bool

            for organ_id in self.all_learned:
                pred_o = (pred       == organ_id) & valid  # bool
                true_o = (mask_batch == organ_id) & valid  # bool

                # --- Cho DSC: union_d = sum(pred) + sum(true) ---
                inter_d[organ_id] += (pred_o & true_o).sum().item()
                union_d[organ_id] += pred_o.sum().item() + true_o.sum().item()

                # --- Cho IoU: union_i = |pred OR true| ---
                inter_i[organ_id] += (pred_o & true_o).sum().item()
                union_i[organ_id] += (pred_o | true_o).sum().item()

        # --- Tinh DSC va IoU tu cac gia tri tich luy ---
        dice_all = {}
        iou_all  = {}
        for organ_id in self.all_learned:
            if union_d[organ_id] > 0:
                dice_all[organ_id] = 2 * inter_d[organ_id] / union_d[organ_id]
            else:
                dice_all[organ_id] = float('nan')

            if union_i[organ_id] > 0:
                iou_all[organ_id] = inter_i[organ_id] / union_i[organ_id]
            else:
                iou_all[organ_id] = float('nan')

        # --- Tach Old / New organs ---
        # C1 = TASK_ORGANS[1] = [6, 7] (Liver + Stomach) -> luon la old organs tu task 2+
        # C2:T = all_learned - C1 = organs hoc sau task 1
        old_organs = [o for o in self.all_learned if o not in self.task_organs]
        dice_old = {o: dice_all[o] for o in old_organs}
        dice_new = {o: dice_all[o] for o in self.task_organs}
        iou_old  = {o: iou_all[o]  for o in old_organs}
        iou_new  = {o: iou_all[o]  for o in self.task_organs}

        return dice_all, dice_old, dice_new, iou_all, iou_old, iou_new

    # ----------------------------------------------------------
    def compute_forgetting(self, current_dice, task_baselines):
        """
        Tinh forgetting DUNG SCOPE: so sanh dice hien tai cua moi organ
        voi dice ngay sau khi organ do duoc hoc xong (baseline).

        FIX: Neu current_dice cua organ la NaN -> SKIP (khong ghi vao dict)
             Ly do: gan 0 se lam forgetting bi overstated voi organs hiem
             (organ khong xuat hien trong eval batch nen NaN, khong phai that su quen)

        Args:
            current_dice   : {organ_id: dice} hien tai
            task_baselines : {organ_id: dice} - dice cua organ ngay sau task
                             ma organ do duoc hoc (la baseline dung scope)
        """
        forgetting = {}
        for organ_id, baseline_dice in task_baselines.items():
            if organ_id in self.task_organs:
                continue  # Khong tinh forgetting cho organs moi nhat
            if np.isnan(baseline_dice):
                continue  # Baseline la NaN -> bo qua

            new_dice = current_dice.get(organ_id, float('nan'))

            # FIX: NaN -> SKIP thay vi gan 0
            # NaN co nghia organ khong xuat hien trong eval batch
            # -> khong co du thong tin de tinh forgetting
            if np.isnan(new_dice):
                continue

            forgetting[organ_id] = baseline_dice - new_dice

        # Chi tinh mean tren cac organs CO du thong tin (khong co NaN)
        valid_fgt = [v for v in forgetting.values() if not np.isnan(v)]
        mean_fgt  = np.mean(valid_fgt) if valid_fgt else float('nan')
        return forgetting, mean_fgt

    # ----------------------------------------------------------
    def save_checkpoint(self, epoch, val_dice, is_best=False,
                        is_best_cil=False, cil_dice=None):
        ewc_lam = self.ewc.ewc_lambda if self.ewc is not None else 0.0
        state = {
            "epoch"          : epoch,
            "task_id"        : self.task_id,
            "method"         : "Online diagonal empirical Fisher EWC-style baseline",
            "model_state"    : self.model.state_dict(),
            "optimizer_state": self.optimizer.state_dict(),
            "scheduler_state": self.scheduler.state_dict() if self.scheduler else None,
            "val_dice"       : float(val_dice),
            "best_val_dice"  : float(self.best_val_dice),
            "best_cil_dice"  : float(self.best_cil_dice),
            "cil_dice"       : float(cil_dice) if cil_dice is not None else None,
            "history"        : self.history,
            "train_cfg"      : TRAIN_CFG,
            "ewc_lambda"     : float(ewc_lam),
            # EWC state (fisher/optpar) saved separately via EWCRegularizer.save()
            # to avoid serialising 200MB of tensors into every checkpoint.
        }
        ckpt_path = os.path.join(
            self.checkpoint_dir,
            f"ewc_task{self.task_id}_epoch{epoch:03d}.pth"
        )
        torch.save(state, ckpt_path)
        if is_best:
            best_path = os.path.join(
                self.checkpoint_dir,
                f"ewc_task{self.task_id}_best.pth"
            )
            torch.save(state, best_path)
            print(f"     Best val checkpoint! Dice={val_dice:.4f} -> {best_path}")
        if is_best_cil:
            best_cil_path = os.path.join(
                self.checkpoint_dir,
                f"ewc_task{self.task_id}_best_cil.pth"
            )
            torch.save(state, best_cil_path)
            print(f"     Best CIL checkpoint! CIL Dice={cil_dice:.4f} "
                  f"-> {best_cil_path}")

    # ----------------------------------------------------------
    def load_prev_task(self, path):
        state = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(state["model_state"])
        src_task  = state["task_id"]
        src_epoch = state["epoch"]
        src_dice  = state.get("best_val_dice", state.get("val_dice", 0))
        print(f"Loaded model from Task {src_task} "
              f"(epoch {src_epoch}, dice={src_dice:.4f})")
        return state

    # ----------------------------------------------------------
    def fit(self, train_loader, val_loader, n_epochs, start_epoch=1,
            cil_val_loader=None, task_baselines=None):
        """
        Train loop chinh.

        Sua doi:
          - evaluate_cil() tra ve 6 gia tri (them iou_all/old/new)
          - Sau final eval: append mIoU(All) vao self.miou_history
          - JSON: luu du schema {dsc, miou, forgetting}
        """
        has_ewc = self.ewc is not None
        print(f"\n{'='*65}")
        print(f"  [EWC v2] Training Task {self.task_id} "
              f"-- Organs: {[ORGAN_NAMES[o] for o in self.task_organs]}")
        print(f"  EWC: {'ACTIVE (λ=' + str(self.ewc.ewc_lambda) + ', normalised/param)' if has_ewc else 'OFF (Task 1)'}")
        print(f"  Epochs: {start_epoch} → {n_epochs}")
        if self.task_id > 1:
            print(f"  All learned organs: {[ORGAN_NAMES[o] for o in self.all_learned]}")
            print(f"  CIL eval every {self.cil_eval_every} epochs")
        print(f"{'='*65}")

        for epoch in range(start_epoch, n_epochs + 1):
            t0 = time.time()

            train_loss, tr_ce, tr_dice, tr_ewc = self.train_epoch(train_loader)
            val_loss, dice_dict = self.validate(val_loader)

            if self.scheduler:
                self.scheduler.step(val_loss)

            current_lr = self.optimizer.param_groups[0]['lr']
            mean_dice  = np.nanmean(list(dice_dict.values()))
            elapsed    = time.time() - t0

            # --- CIL evaluation moi N epoch (detect forgetting mid-training) ---
            cil_overall = None
            is_best_cil = False
            do_cil = (cil_val_loader is not None
                      and self.task_id > 1
                      and (epoch % self.cil_eval_every == 0
                           or epoch == n_epochs))
            if do_cil:
                # Goi evaluate_cil moi, nhan 6 gia tri (them iou)
                (dice_all, dice_old, dice_new,
                 iou_all,  iou_old,  iou_new) = self.evaluate_cil(cil_val_loader, split="val")

                all_valid = [v for v in dice_all.values() if not np.isnan(v)]
                cil_overall = np.mean(all_valid) if all_valid else 0.0

                if cil_overall > self.best_cil_dice:
                    self.best_cil_dice = cil_overall
                    is_best_cil = True

                old_valid = [v for v in dice_old.values() if not np.isnan(v)]
                old_mean  = np.mean(old_valid) if old_valid else 0.0

                if task_baselines:
                    _, mean_fgt = self.compute_forgetting(dice_all, task_baselines)
                else:
                    mean_fgt = float('nan')

                fgt_str = f"{mean_fgt:+.4f}" if not np.isnan(mean_fgt) else "N/A"
                print(f"  >> CIL: overall={cil_overall:.4f} | "
                      f"old_organs={old_mean:.4f} | "
                      f"forgetting={fgt_str}"
                      f"{' [NEW BEST CIL]' if is_best_cil else ''}")

            # EWC ratio: ewc_loss / seg_loss — key diagnostic for EWC health.
            # Target: ratio ≈ 0.1–1.0 (EWC active but not dominating).
            # If ratio < 0.01 → lambda too small or Fisher too small.
            seg_ref   = max(train_loss - tr_ewc, 1e-8)
            ewc_ratio = tr_ewc / seg_ref if tr_ewc > 0 else 0.0

            self.history["train_loss"].append(float(train_loss))
            self.history["val_loss"].append(float(val_loss))
            self.history["val_dice"].append(float(mean_dice))
            self.history["lr"].append(float(current_lr))
            self.history["ewc_loss"].append(float(tr_ewc))
            self.history["ewc_ratio"].append(float(ewc_ratio))
            self.history["cil_dice"].append(
                float(cil_overall) if cil_overall is not None else None)

            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_diag_str = ""
            if has_ewc:
                fisher_diag = self.ewc.fisher_diagnostics()
                drift_diag = self.ewc.parameter_drift_norms(self.model)
                inactive = (tr_ewc <= 1e-12) or (ewc_ratio < 1e-4)
                warn = []
                if fisher_diag["fisher_collapse"]:
                    warn.append("FISHER_COLLAPSE")
                if fisher_diag["fisher_explosion"]:
                    warn.append("FISHER_EXPLOSION")
                if inactive:
                    warn.append("INACTIVE_EWC")
                ewc_diag_str = (f" | Fisher mean/max/min="
                                f"{fisher_diag['fisher_mean']:.3e}/"
                                f"{fisher_diag['fisher_max']:.3e}/"
                                f"{fisher_diag['fisher_min']:.3e}"
                                f" | driftL2={drift_diag['drift_l2']:.3e}"
                                f" | flags={','.join(warn) if warn else 'OK'}")
            ewc_str = (f"EWC={tr_ewc:.4f} [ratio={ewc_ratio:.3f}]"
                       if has_ewc else "EWC=OFF")
            print(f"\n  Epoch {epoch:3d}/{n_epochs} "
                  f"({elapsed:.0f}s) | LR={current_lr:.2e}")
            print(f"  Train: loss={train_loss:.4f} "
                  f"(CE={tr_ce:.4f}, Dice={tr_dice:.4f}, {ewc_str})"
                  f"{ewc_diag_str}")
            print(f"  [VAL] loss={val_loss:.4f} | Mean Dice={mean_dice:.4f}")

            format_dice_table(dice_dict, self.task_organs, epoch=epoch)
            log_vram(f"epoch {epoch}")

            is_best = mean_dice > self.best_val_dice
            if is_best:
                self.best_val_dice = mean_dice
            if epoch % self.save_every == 0 or is_best or is_best_cil:
                self.save_checkpoint(epoch, mean_dice, is_best,
                                     is_best_cil, cil_overall)

        self._save_log()

        print(f"\n{'='*65}")
        print(f"  [EWC] Training Task {self.task_id} hoan tat!")
        print(f"  Best Val Dice  : {self.best_val_dice:.4f}")
        if self.task_id > 1:
            print(f"  Best CIL Dice  : {self.best_cil_dice:.4f}")
        print(f"{'='*65}")

        # ---- Final CIL Evaluation (dung best_cil checkpoint neu co) ----
        if cil_val_loader is not None:
            if self.task_id > 1 and self.best_cil_dice > 0:
                best_cil_path = os.path.join(
                    self.checkpoint_dir,
                    f"ewc_task{self.task_id}_best_cil.pth"
                )
                if os.path.exists(best_cil_path):
                    state = torch.load(best_cil_path, map_location=self.device,
                                       weights_only=False)
                    self.model.load_state_dict(state["model_state"])
                    print(f"  Loaded best CIL checkpoint (epoch {state['epoch']}) "
                          f"for final evaluation")

            print(f"\n  Running final CIL evaluation...")
            (dice_all, dice_old, dice_new,
             iou_all,  iou_old,  iou_new) = self.evaluate_cil(cil_val_loader, split="val")

            # --- Append mIoU(All) vao history de tinh avg_miou ---
            # avg_miou tich hop hieu nang toan bo qua trinh CIL, khong chi cuoi
            iou_all_vals = [v for v in iou_all.values() if not np.isnan(v)]
            miou_all_this_task = np.mean(iou_all_vals) if iou_all_vals else float('nan')
            if not np.isnan(miou_all_this_task):
                self.miou_history.append(miou_all_this_task)
            avg_miou = np.mean(self.miou_history) if self.miou_history else float('nan')
            print(f"  mIoU(All) task {self.task_id}: {miou_all_this_task:.4f} "
                  f"| avg_mIoU so far: {avg_miou:.4f}")

            # --- Tinh forgetting cho JSON ---
            fgt_dict, mean_fgt = ({}, float('nan'))
            if task_baselines:
                fgt_dict, mean_fgt = self.compute_forgetting(dice_all, task_baselines)

            overall = self._print_cil_report(
              dice_all, dice_old, dice_new,
              iou_all,  iou_old,  iou_new,
              avg_miou=avg_miou,
              fgt_dict=fgt_dict,
              mean_fgt=mean_fgt
            )

            # --- Tinh cac chi so DSC va mIoU phan nhom C1 / C2:T / All ---
            # C1 = TASK_ORGANS[1] = [6, 7]  (Liver + Stomach) - luon la nhom cu nhat
            C1_organs = TASK_ORGANS[1]
            C2T_organs = [o for o in self.all_learned if o not in C1_organs]

            # DSC
            dsc_c1  = np.nanmean([dice_all.get(o, float('nan')) for o in C1_organs])
            dsc_c2t = np.nanmean([dice_all.get(o, float('nan')) for o in C2T_organs]) if C2T_organs else float('nan')
            dsc_all = np.nanmean([v for v in dice_all.values() if not np.isnan(v)]) if dice_all else float('nan')

            # mIoU
            miou_c1  = np.nanmean([iou_all.get(o, float('nan')) for o in C1_organs])
            miou_c2t = np.nanmean([iou_all.get(o, float('nan')) for o in C2T_organs]) if C2T_organs else float('nan')

            # --- Luu JSON theo schema moi ---
            # {
            #   task_id, method, ewc_lambda,
            #   dsc: {old, new, all, per_organ},
            #   miou: {c1, c2t, all, avg, per_organ},
            #   forgetting: {per_organ, mean}
            # }
            ewc_lam = self.ewc.ewc_lambda if self.ewc is not None else 0.0
            cil_results = {
                "method"        : "Online diagonal empirical Fisher EWC-style baseline",
                "task_id"       : self.task_id,
                "ewc_lambda"    : float(ewc_lam),
                "fisher_batch"  : FISHER_BATCH_SIZE,
                "fisher_samples": FISHER_SAMPLES,
                "best_val_dice" : _safe_float(self.best_val_dice),
                "best_cil_dice" : _safe_float(self.best_cil_dice),
                "dsc": {
                    "old"      : _safe_float(dsc_c1),
                    "new"      : _safe_float(dsc_c2t),
                    "all"      : _safe_float(dsc_all),
                    "per_organ": {
                        ORGAN_NAMES[k]: _safe_float(v)
                        for k, v in dice_all.items() if not np.isnan(v)
                    },
                },
                "miou": {
                    "c1"       : _safe_float(miou_c1),
                    "c2t"      : _safe_float(miou_c2t),
                    "all"      : _safe_float(miou_all_this_task),
                    "avg"      : _safe_float(avg_miou),
                    "per_organ": {
                        ORGAN_NAMES[k]: _safe_float(v)
                        for k, v in iou_all.items() if not np.isnan(v)
                    },
                },
                "forgetting": {
                    "per_organ": {
                        ORGAN_NAMES[k]: _safe_float(v)
                        for k, v in fgt_dict.items()
                    },
                    "mean": _safe_float(mean_fgt),
                },
            }

            cil_path = os.path.join(
                self.log_dir, f"ewc_task{self.task_id}_cil_results.json"
            )
            with open(cil_path, 'w') as f:
                json.dump(cil_results, f, indent=2, ensure_ascii=False)
            print(f"   CIL results saved: {cil_path}")

            return dice_all, iou_all

    # ----------------------------------------------------------
    def _print_cil_report(self, dice_all, dice_old, dice_new,
                           iou_all, iou_old, iou_new,
                           avg_miou=None, task_baselines=None,
                           fgt_dict=None, mean_fgt=None):
        """
        In bao cao CIL day du:
          - He thong 1 (DSC): Old / New / All  +  IoU tuong ung
          - He thong 2 (mIoU): C1 / C2:T / All / avg_miou
          - Forgetting: theo organ, skip NaN (khong gan 0)
        """
        print(f"\n{'='*75}")
        print(f"  [EWC] CIL EVALUATION REPORT - Sau Task {self.task_id}")
        print(f"{'='*75}")

        # ---- He thong 1: DSC ----
        # Nhom C1 = TASK_ORGANS[1] = [6, 7]  -> do kha nang GIU kien thuc cu
        # Nhom C2:T = cac organs hoc sau Task 1 -> do HOC moi
        C1_organs  = TASK_ORGANS[1]  # [6, 7]
        C2T_organs = [o for o in self.all_learned if o not in C1_organs]

        print(f"\n  [DSC] He thong 1 - Dice Similarity Coefficient")
        print(f"  {'Nhom':12s}  {'Organ':30s}  {'DSC':>8}  {'IoU':>8}")
        print(f"  {'-'*64}")

        # --- DSC Old: C1 organs (Liver + Stomach) ---
        dsc_c1_vals = []
        iou_c1_vals = []
        for organ_id in C1_organs:
            d = dice_all.get(organ_id, float('nan'))
            u = iou_all.get(organ_id, float('nan'))
            name = ORGAN_NAMES.get(organ_id, f"Organ {organ_id}")
            d_str = f"{d:.4f}" if not np.isnan(d) else "  NaN "
            u_str = f"{u:.4f}" if not np.isnan(u) else "  NaN "
            print(f"  {'DSC Old':12s}  {name:30s}  {d_str:>8}  {u_str:>8}")
            if not np.isnan(d): dsc_c1_vals.append(d)
            if not np.isnan(u): iou_c1_vals.append(u)

        dsc_old_mean = np.mean(dsc_c1_vals) if dsc_c1_vals else float('nan')
        iou_c1_mean  = np.mean(iou_c1_vals) if iou_c1_vals else float('nan')
        print(f"  {'':12s}  {'** DSC Old (mean C1)':30s}  {dsc_old_mean:8.4f}  {iou_c1_mean:8.4f}")

        # --- DSC New: C2:T organs ---
        dsc_new_vals = []
        iou_new_vals = []
        for organ_id in C2T_organs:
            d = dice_all.get(organ_id, float('nan'))
            u = iou_all.get(organ_id, float('nan'))
            name = ORGAN_NAMES.get(organ_id, f"Organ {organ_id}")
            d_str = f"{d:.4f}" if not np.isnan(d) else "  NaN "
            u_str = f"{u:.4f}" if not np.isnan(u) else "  NaN "
            print(f"  {'DSC New':12s}  {name:30s}  {d_str:>8}  {u_str:>8}")
            if not np.isnan(d): dsc_new_vals.append(d)
            if not np.isnan(u): iou_new_vals.append(u)

        dsc_new_mean = np.mean(dsc_new_vals) if dsc_new_vals else float('nan')
        iou_c2t_mean = np.mean(iou_new_vals) if iou_new_vals else float('nan')
        if C2T_organs:
            print(f"  {'':12s}  {'** DSC New (mean C2:T)':30s}  {dsc_new_mean:8.4f}  {iou_c2t_mean:8.4f}")

        # --- DSC All ---
        all_dsc_vals = [v for v in dice_all.values() if not np.isnan(v)]
        all_iou_vals = [v for v in iou_all.values()  if not np.isnan(v)]
        dsc_all_mean = np.mean(all_dsc_vals) if all_dsc_vals else float('nan')
        iou_all_mean = np.mean(all_iou_vals) if all_iou_vals else float('nan')
        print(f"  {'='*64}")
        print(f"  {'DSC All':12s}  {'** Mean (C1:T, tat ca organs)':30s}  "
              f"{dsc_all_mean:8.4f}  {iou_all_mean:8.4f}")

        # ---- He thong 2: mIoU ----
        print(f"\n  [mIoU] He thong 2 - mean Intersection over Union")
        print(f"  {'-'*50}")
        iou_c1_str  = f"{iou_c1_mean:.4f}"  if not np.isnan(iou_c1_mean)  else "N/A"
        iou_c2t_str = f"{iou_c2t_mean:.4f}" if not np.isnan(iou_c2t_mean) else "N/A"
        iou_all_str = f"{iou_all_mean:.4f}" if not np.isnan(iou_all_mean) else "N/A"
        avg_str     = f"{avg_miou:.4f}"     if (avg_miou is not None and not np.isnan(avg_miou)) else "N/A"

        print(f"  mIoU C1        (Liver+Stomach, do rigidity)    : {iou_c1_str}")
        print(f"  mIoU C2:T      (organs hoc sau, do plasticity) : {iou_c2t_str}")
        print(f"  mIoU All       (toan bo C1:T, tong the)        : {iou_all_str}")
        print(f"  avg mIoU       (trung binh qua tat ca tasks)   : {avg_str}")
        print(f"  (avg mIoU tich hop hieu nang TOAN BO qua trinh CIL, khong chi task cuoi)")

        # ---- Forgetting ----
        # FIX-02: Dung fgt_dict/mean_fgt da duoc tinh 1 lan trong fit()
        # KHONG goi self.compute_forgetting() lai lan 2 — tranh diverge JSON vs output
        if fgt_dict is not None and len(fgt_dict) > 0:
            print(f"\n  [FORGETTING] So voi baseline ngay sau khi hoc:")
            print(f"  {'Organ':30s}  {'Forgetting':>12}  {'Note':>12}")
            print(f"  {'-'*58}")
            for organ_id, fgt in fgt_dict.items():
                name   = ORGAN_NAMES.get(organ_id, f"Organ {organ_id}")
                marker = "<-- SEVERE" if fgt > 0.2 else ""
                print(f"  {name:30s}  {fgt:+12.4f}  {marker}")
            fgt_str = f"{mean_fgt:+.4f}" if (mean_fgt is not None and not np.isnan(mean_fgt)) else "N/A"
            print(f"  {'Mean Forgetting':30s}  {fgt_str}")
        elif fgt_dict is not None:
            print(f"\n  [FORGETTING] Khong co du lieu (tat ca organs bi NaN -> skip)")

        print(f"\n  {'='*75}")
        return dsc_all_mean

    # ----------------------------------------------------------
    def _save_log(self):
        import csv
        log_path = os.path.join(self.log_dir, f"ewc_task{self.task_id}_history.csv")
        fields = ["epoch","train_loss","val_loss","val_dice",
                  "lr","ewc_loss","ewc_ratio","cil_dice"]
        with open(log_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            for i, (tl, vl, vd, lr, ew, er, cd) in enumerate(
                    zip(self.history["train_loss"], self.history["val_loss"],
                        self.history["val_dice"],   self.history["lr"],
                        self.history["ewc_loss"],   self.history["ewc_ratio"],
                        self.history["cil_dice"]), 1):
                writer.writerow({"epoch": i, "train_loss": tl, "val_loss": vl,
                                 "val_dice": vd, "lr": lr, "ewc_loss": ew,
                                 "ewc_ratio": er,
                                 "cil_dice": cd if cd is not None else ""})
        print(f"   Log → {log_path}")


print("EWCTrainer (reviewer-ready v2) ready!")
print("  - AMP: torch.amp.autocast / torch.amp.GradScaler (non-deprecated)")
print("  - EWC via EWCRegularizer (vectorised, normalised by num_params)")
print("  - ewc_ratio logged per epoch (ewc_loss / seg_loss)")
print("  - JSON: _safe_float — no np.float64 leakage")
print("  - save_checkpoint: no fisher tensor serialisation (managed by EWCRegularizer)")









## Step 5 — Train All Tasks voi EWC (FIXED)

Quy trinh:
```
Task 1: Load task1_best.pth (da train o Notebook 02)
        → Compute Fisher (FULL mask, khong remap)
        → Luu old_params + Fisher ra DISK
        → Save ewc_task1_cil_results.json (baseline)

Task 2-4: Load model + EWC state tu disk (chong Colab restart)
          → Train 50 epochs voi CIL eval moi 10 epoch
          → Best checkpoint theo CIL Dice (khong chi val_dice)
          → Online EWC Fisher accumulation (decay=0.9)
          → Save EWC state ra disk sau moi task
          → Per-task baseline tracking cho forgetting
```

Fixes applied:
- Fisher dung full mask (14 classes) thay vi task-remapped
- Fisher + old_params luu ra disk, load lai khi resume
- Resume logic ho tro STARTING_TASK = 2, 3, hoac 4
- Online EWC voi decay chong Fisher tang vo han
- CIL eval mid-training de detect forgetting
- Best checkpoint theo CIL overall Dice

In [ ]:
# ============================================================
# Data split — 3-way Train / Val / Test
# ============================================================
# [TRAIN] train split is used for optimization and empirical Fisher estimation.
# [VAL] validation split is used only for scheduler/checkpoint selection/tuning.
# [TEST] test split is used only for final paper-reportable metrics.
# ============================================================
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

SPLIT_PATH = f"{DRIVE_ROOT}/data/processed/volume_split_3way.json"
all_vol_ids = sorted(set(r["volume_id"] for r in metadata))

if os.path.exists(SPLIT_PATH):
    with open(SPLIT_PATH) as f:
        _split = json.load(f)
    train_vol_ids = set(_split["train_vol_ids"])
    val_vol_ids   = set(_split["val_vol_ids"])
    test_vol_ids  = set(_split["test_vol_ids"])
    print(f"Loaded 3-way split: {len(train_vol_ids)} train / "
          f"{len(val_vol_ids)} val / {len(test_vol_ids)} test volumes")
else:
    print("Creating 3-way split (train/val/test)...")
    set_global_seed(TRAIN_CFG["seed"])
    shuffled = list(all_vol_ids)
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(n * TRAIN_CFG["train_ratio"]))
    n_val   = max(1, int(n * TRAIN_CFG["val_ratio"]))
    train_vol_ids = set(shuffled[:n_train])
    val_vol_ids   = set(shuffled[n_train:n_train + n_val])
    test_vol_ids  = set(shuffled[n_train + n_val:])
    with open(SPLIT_PATH, 'w') as f:
        json.dump({"train_vol_ids": sorted(train_vol_ids),
                   "val_vol_ids"  : sorted(val_vol_ids),
                   "test_vol_ids" : sorted(test_vol_ids),
                   "seed"         : TRAIN_CFG["seed"],
                   "ratios"       : [TRAIN_CFG["train_ratio"],
                                     TRAIN_CFG["val_ratio"],
                                     TRAIN_CFG["test_ratio"]],
                   "created_by"   : "NB03-online-ewc-style"}, f, indent=2)

assert train_vol_ids.isdisjoint(val_vol_ids), "LEAKAGE: train/val overlap"
assert train_vol_ids.isdisjoint(test_vol_ids), "LEAKAGE: train/test overlap"
assert val_vol_ids.isdisjoint(test_vol_ids), "LEAKAGE: val/test overlap"
assert len(test_vol_ids) > 0, "Invalid split: empty TEST set"

def tag_records(records, split):
    tagged = []
    for r in records:
        rr = dict(r)
        rr["split"] = split
        tagged.append(rr)
    return tagged

def assert_records_split(records, split):
    bad = [r.get("volume_id") for r in records if r.get("split") != split]
    assert not bad, f"Wrong split records detected: expected {split}, examples={bad[:5]}"
    if split == "test":
        assert all(r["volume_id"] in test_vol_ids for r in records), "TEST records include non-test volumes"
        assert not any(r["volume_id"] in val_vol_ids for r in records), "VALIDATION LEAKAGE in final TEST records"
        assert not any(r["volume_id"] in train_vol_ids for r in records), "TRAIN LEAKAGE in final TEST records"

train_records_all = tag_records([r for r in metadata if r["volume_id"] in train_vol_ids], "train")
val_records_all   = tag_records([r for r in metadata if r["volume_id"] in val_vol_ids], "val")
test_records_all  = tag_records([r for r in metadata if r["volume_id"] in test_vol_ids], "test")

assert_records_split(train_records_all, "train")
assert_records_split(val_records_all, "val")
assert_records_split(test_records_all, "test")

print(f"\nTotal slices : {len(metadata)}")
print(f"[TRAIN] volumes={len(train_vol_ids)} slices={len(train_records_all)}")
print(f"[VAL]   volumes={len(val_vol_ids)} slices={len(val_records_all)} checkpoint selection only")
print(f"[TEST]  volumes={len(test_vol_ids)} slices={len(test_records_all)} final metrics only")
print("[ASSERTION PASSED] No train/val/test volume overlap; TEST isolated for final metrics.")

val_cil_dataset = BTCVDataset(
    records    = val_records_all,
    images_dir = IMAGES_2D_DIR,
    masks_dir  = MASKS_2D_DIR,
    task_id    = None,
    augment    = False,
)
val_cil_loader = DataLoader(
    val_cil_dataset, batch_size=TRAIN_CFG["batch_size"],
    shuffle=False, num_workers=TRAIN_CFG["num_workers"], pin_memory=True,
)

test_cil_dataset = BTCVDataset(
    records    = test_records_all,
    images_dir = IMAGES_2D_DIR,
    masks_dir  = MASKS_2D_DIR,
    task_id    = None,
    augment    = False,
)
test_cil_loader = DataLoader(
    test_cil_dataset, batch_size=TRAIN_CFG["batch_size"],
    shuffle=False, num_workers=TRAIN_CFG["num_workers"], pin_memory=True,
)

assert_records_split(test_records_all, "test")
print(f"\n[VAL]  CIL loader: {len(val_cil_dataset)} slices  (checkpoint selection)")
print(f"[TEST] CIL loader: {len(test_cil_dataset)} slices  (FINAL REPORT ONLY)")



In [ ]:
# ── DEBUG: Verify Fisher state in existing checkpoints ───────────────────
# PURPOSE: Diagnostic only — run after training to confirm Fisher was saved.
# Does NOT affect training or reproducibility settings.
# NOTE: EWC state is stored in ewc_state_after_task{N}.pth (via EWCRegularizer.save),
#       NOT embedded in model checkpoints (by design, to avoid 200MB per checkpoint).
#       This cell checks the separate EWC state files.

print("=== Kiểm tra Fisher Information (EWC state files) ===")

for task_id in range(1, 5):
    ewc_state_path = f"{CHECKPOINT_DIR}/ewc_state_after_task{task_id}.pth"
    if not os.path.exists(ewc_state_path):
        print(f"  Task {task_id}: ewc_state file MISSING — run training first")
        continue

    ewc_state = torch.load(ewc_state_path, map_location="cpu", weights_only=False)
    fisher     = ewc_state.get("fisher", {})
    old_params = ewc_state.get("optpar", {})

    if fisher:
        fisher_sum  = sum(v.sum().item() for v in fisher.values())
        fisher_mean = sum(v.mean().item() for v in fisher.values()) / max(len(fisher), 1)
        fisher_max  = max(v.max().item() for v in fisher.values())
        # ASSERTION: Fisher must be positive and finite (publication requirement)
        assert fisher_sum > 0, f"Task {task_id}: Fisher sum=0 — EWC is inactive!"
        assert not any(v.isnan().any() for v in fisher.values()), f"Task {task_id}: Fisher has NaN!"
        print(f"  Task {task_id}: fisher_sum={fisher_sum:.4f}  "
              f"fisher_mean={fisher_mean:.4e}  fisher_max={fisher_max:.4e}  "
              f"learned={sorted(ewc_state.get('learned_classes', []))}")
    else:
        print(f"  Task {task_id}: fisher dict EMPTY — check EWCRegularizer.save()")

    del ewc_state
print("=== Fisher check complete ===")


In [ ]:
# ── EWC State Architecture (v2) ──────────────────────────────────────────
# EWC state (Fisher + old_params) is stored SEPARATELY from model checkpoints:
#
#   checkpoints_ewc_v2/
#     ewc_state_after_task1.pth   ← Fisher + optpar after Task 1 (via EWCRegularizer.save)
#     ewc_state_after_task2.pth   ← EMA-accumulated Fisher after Task 2
#     ewc_state_after_task3.pth   ← EMA-accumulated Fisher after Task 3
#     ewc_state_after_taskN.pth   ← (auto-saved after each task)
#     ewc_task{N}_best.pth        ← model + optimizer (NO Fisher embedded)
#     ewc_task{N}_best_cil.pth    ← model selected by CIL Dice on VAL set
#
# WHY separate storage:
#   - Fisher tensors are ~200MB per task (same size as model)
#   - Embedding in every checkpoint wastes 1-2GB per task
#   - EWCRegularizer.save/load handles device management automatically
#
# TO RESUME training from Task N:
#   1. Set STARTING_TASK = N
#   2. The resume block loads ewc_state_after_task{N-1}.pth automatically
#   3. No manual patching needed
#
# NOTE: Fresh runs must use the saved EWC state files directly.
#       This notebook does not modify checkpoints after training.

print("EWC state architecture: Fisher stored in ewc_state_after_task{N}.pth")
print("No patch needed for fresh runs.")







In [ ]:
# ============================================================
# Task 1: Load checkpoint + compute diagonal empirical Fisher
# ============================================================
# Key changes:
#   - EWCRegularizer replaces manual fisher + old_params dicts
#   - task_id=1 mask (task-specific Fisher, not full 14-class)
#   - FISHER_BATCH_SIZE=1 (per-sample unbiased gradient)
#   - CIL baseline on TEST loader only for final evaluation; VAL remains checkpoint selection only
#   - update_learned_classes() before Fisher so channel mask is correct
# ============================================================

print("="*65)
print("  TASK 1: Load checkpoint + Compute Fisher (v2 — task-specific)")
print("="*65)

# 1. Load model Task 1 (trained in Notebook 02)
model = build_unet(
    num_classes = TRAIN_CFG["num_classes"],
    encoder     = TRAIN_CFG["encoder"],
    pretrained  = TRAIN_CFG["pretrained"],
).to(DEVICE)

# Task 1 checkpoint is in checkpoints_baseline (trained in NB02)
task1_ckpt_path = f"{DRIVE_ROOT}/checkpoints_baseline/task1_best.pth"
ckpt = torch.load(task1_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
t1_dice = ckpt.get('best_val_dice', ckpt.get('val_dice', 0))
print(f"Loaded Task 1: epoch={ckpt['epoch']}, dice={t1_dice:.4f}")

# 2. Initialise EWCRegularizer
ewc_reg = EWCRegularizer(
    model       = model,
    num_classes = TRAIN_CFG["num_classes"],
    device      = DEVICE,
    ewc_lambda  = EWC_LAMBDA,
    decay       = EWC_FISHER_DECAY,
).to(DEVICE)

# Register Task 1 organs BEFORE computing Fisher so channel mask is correct.
# Finetune ablation skips Fisher because no EWC-style regularization is used.
ewc_state_path = None
if USE_EWC_REGULARIZATION:
    ewc_reg.update_learned_classes(TASK_ORGANS[1])

    # 3. Compute task-specific Fisher (FIX-2: task_id=1, not None)
    task1_slices = get_slices_for_task(metadata, task_id=1)
    task1_train  = [r for r in train_records_all if set(TASK_ORGANS[1]) & set(r["organs_present"])]
    assert_records_split(task1_train, "train")

    fisher_dataset = BTCVDataset(
        records    = task1_train,
        images_dir = IMAGES_2D_DIR,
        masks_dir  = MASKS_2D_DIR,
        task_id    = 1,
        augment    = False,
    )
    fisher_loader = DataLoader(
        fisher_dataset,
        batch_size  = FISHER_BATCH_SIZE,
        shuffle     = True,
        num_workers = TRAIN_CFG["num_workers"],
    )
    print(f"\nFisher config: task_id=1 mask, batch={FISHER_BATCH_SIZE}, "
          f"samples={FISHER_SAMPLES}")

    ewc_reg.compute_and_set_fisher(
        model       = model,
        dataloader  = fisher_loader,
        task_organs = TASK_ORGANS[1],
        num_samples = FISHER_SAMPLES,
    )

    ewc_state_path = f"{CHECKPOINT_DIR}/ewc_state_after_task1.pth"
    ewc_reg.save(ewc_state_path, task_id=1)
else:
    ewc_reg = None
    print("\n[ABLATION] Finetune baseline: Task 1 Fisher computation skipped.")

# 5. CIL baseline on TEST loader (prevents data leakage into model selection)
print(f"\nTask 1 CIL baseline [TEST set, {len(test_cil_dataset)} slices]:")
task1_cil_dice = {}
task1_cil_iou  = {}
inter_d = defaultdict(float); union_d = defaultdict(float)
inter_i = defaultdict(float); union_i = defaultdict(float)
model.eval()
with torch.no_grad():
    for ct_b, mask_b in test_cil_loader:
        ct_b   = ct_b.to(DEVICE)
        mask_b = mask_b.to(DEVICE)
        pred   = torch.argmax(model(ct_b), dim=1)
        valid  = (mask_b != IGNORE_INDEX)
        for oid in TASK_ORGANS[1]:
            p_o = (pred   == oid) & valid
            t_o = (mask_b == oid) & valid
            inter_d[oid] += (p_o & t_o).sum().item()
            union_d[oid] += p_o.sum().item() + t_o.sum().item()
            inter_i[oid] += (p_o & t_o).sum().item()
            union_i[oid] += (p_o | t_o).sum().item()

for oid in TASK_ORGANS[1]:
    task1_cil_dice[oid] = (2 * inter_d[oid] / union_d[oid]
                           if union_d[oid] > 0 else float('nan'))
    task1_cil_iou[oid]  = (inter_i[oid] / union_i[oid]
                           if union_i[oid] > 0 else float('nan'))
    d_s = f"{task1_cil_dice[oid]:.4f}" if not np.isnan(task1_cil_dice[oid]) else "NaN"
    u_s = f"{task1_cil_iou[oid]:.4f}"  if not np.isnan(task1_cil_iou[oid])  else "NaN"
    print(f"  {ORGAN_NAMES[oid]:30s}  DSC={d_s}  IoU={u_s}")

task1_miou_all = float(np.nanmean(list(task1_cil_iou.values())))
task1_mdsc_all = float(np.nanmean(list(task1_cil_dice.values())))

# 6. Save Task 1 JSON (schema consistent with Tasks 2-4)
task1_cil_results = {
    "method"       : "Online diagonal empirical Fisher EWC-style baseline",
    "task_id"      : 1,
    "ewc_lambda"   : float(EWC_LAMBDA),
    "fisher_batch" : FISHER_BATCH_SIZE,
    "best_val_dice": _safe_float(t1_dice),
    "dsc": {
        "old"      : None,
        "new"      : _safe_float(task1_mdsc_all),
        "all"      : _safe_float(task1_mdsc_all),
        "per_organ": {ORGAN_NAMES[k]: _safe_float(v)
                      for k, v in task1_cil_dice.items() if not np.isnan(v)},
    },
    "miou": {
        "c1"       : _safe_float(task1_miou_all),
        "c2t"      : None,
        "all"      : _safe_float(task1_miou_all),
        "avg"      : _safe_float(task1_miou_all),
        "per_organ": {ORGAN_NAMES[k]: _safe_float(v)
                      for k, v in task1_cil_iou.items() if not np.isnan(v)},
    },
    "forgetting": {"per_organ": {}, "mean": None},
}
task1_cil_path = f"{LOG_DIR}/ewc_task1_cil_results.json"
with open(task1_cil_path, 'w') as f:
    json.dump(task1_cil_results, f, indent=2, ensure_ascii=False)
print(f"\nTask 1 CIL results saved: {task1_cil_path}")

# 7. Initialise mIoU history and organ baselines (persisted to disk)
MIOU_HISTORY_PATH = f"{LOG_DIR}/global_miou_history.json"
BASELINES_PATH    = f"{LOG_DIR}/per_organ_baselines.json"

global_miou_history = ([float(task1_miou_all)]
                        if not np.isnan(task1_miou_all) else [])
save_miou_history(global_miou_history, MIOU_HISTORY_PATH)

task_baselines = update_organ_baselines({}, task1_cil_dice, TASK_ORGANS[1],
                                        BASELINES_PATH)

print(f"\nmIoU(All) Task 1 [TEST]: {task1_miou_all:.4f}")
print(f"Organ baselines init: {task_baselines}")
log_vram("after Fisher Task 1")








In [ ]:
# STARTING_TASK: set to 2 for a fresh run.
# Change to 3 or 4 to resume from a previous interrupted run.
STARTING_TASK = 2

# Paths defined in task1-fisher cell; re-stated here for resume safety
BASELINES_PATH    = f"{LOG_DIR}/per_organ_baselines.json"
MIOU_HISTORY_PATH = f"{LOG_DIR}/global_miou_history.json"

# save_organ_baselines / load_organ_baselines / update_organ_baselines /
# save_miou_history / load_miou_history are defined in the EWCRegularizer cell.

# ============================================================
# Initialise EWC state and baselines
# ============================================================
if STARTING_TASK == 1:
    # Fresh run: ewc_reg and task_baselines already in RAM from task1-fisher cell
    pass  # ewc_reg, task_baselines, global_miou_history set in task1-fisher cell

else:  # Resume from STARTING_TASK > 1
    prev_task_id = STARTING_TASK - 1

    # --- Load model tu best checkpoint cua task truoc ---
    prev_ckpt = f"{CHECKPOINT_DIR}/ewc_task{prev_task_id}_best_cil.pth"
    if not os.path.exists(prev_ckpt):
        prev_ckpt = f"{CHECKPOINT_DIR}/ewc_task{prev_task_id}_best.pth"
    if not os.path.exists(prev_ckpt) and prev_task_id == 1:
        # Task 1 duoc train o Notebook 02, dung ten goc
        prev_ckpt = f"{CHECKPOINT_DIR}/task1_best.pth"
    if not os.path.exists(prev_ckpt):
        raise FileNotFoundError(
            f"Khong tim thay checkpoint Task {prev_task_id}. "
            f"Kiem tra lai: {prev_ckpt}")

    ckpt_prev = torch.load(prev_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_prev["model_state"])
    print(f"  Resumed MODEL from: {prev_ckpt}")
    print(f"  (Task {ckpt_prev['task_id']}, epoch {ckpt_prev['epoch']}, "
          f"best_val_dice={ckpt_prev.get('best_val_dice', 'N/A')})")

    # --- Load Fisher + reference params only for EWC-style regularization ---
    if USE_EWC_REGULARIZATION:
        ewc_state_path = f"{CHECKPOINT_DIR}/ewc_state_after_task{prev_task_id}.pth"
        ewc_reg = EWCRegularizer.load(ewc_state_path, model=model,
                                      num_classes=TRAIN_CFG["num_classes"],
                                      device=DEVICE)
    else:
        ewc_reg = None
        print("  [ABLATION] Finetune baseline: no EWC state loaded.")

    # Load task_baselines from disk
    task_baselines = load_organ_baselines(BASELINES_PATH)
    if task_baselines:
        print(f"  Organ baselines loaded: {len(task_baselines)} organs")
    else:
        print("  WARNING: per_organ_baselines.json missing — forgetting may be incorrect")

    global_miou_history = load_miou_history(MIOU_HISTORY_PATH)
    print(f"  Resumed mIoU history: {[round(x,4) for x in global_miou_history]}")

print(f"\nSetup: STARTING_TASK={STARTING_TASK}, "
      f"baselines={len(task_baselines)} organs")
print(f"Baselines: { {ORGAN_NAMES[k]: round(v,4) for k,v in task_baselines.items()} }")

# ============================================================
# Main training loop: Tasks 2 → 3 → 4
# ============================================================
for task_id in range(STARTING_TASK, 5):

    print(f"\n\n{'#'*70}")
    print(f"#  Online diagonal empirical Fisher EWC-style baseline — TASK {task_id}")
    print(f"{'#'*70}")

    # Register this task's organs so channel mask is updated BEFORE training
    if USE_EWC_REGULARIZATION:
        ewc_reg.update_learned_classes(TASK_ORGANS[task_id])

    # Data for this task
    task_slices   = get_slices_for_task(metadata, task_id)
    train_records = [r for r in train_records_all if set(TASK_ORGANS[task_id]) & set(r["organs_present"])]
    val_records   = [r for r in val_records_all if set(TASK_ORGANS[task_id]) & set(r["organs_present"])]
    assert_records_split(train_records, "train")
    assert_records_split(val_records, "val")

    train_dataset = BTCVDataset(
        records=train_records, images_dir=IMAGES_2D_DIR,
        masks_dir=MASKS_2D_DIR, task_id=task_id, augment=True)
    val_dataset = BTCVDataset(
        records=val_records, images_dir=IMAGES_2D_DIR,
        masks_dir=MASKS_2D_DIR, task_id=task_id, augment=False)

    g = torch.Generator()
    g.manual_seed(TRAIN_CFG["seed"] + task_id)

    train_loader = DataLoader(
        train_dataset, batch_size=TRAIN_CFG["batch_size"],
        shuffle=True, num_workers=TRAIN_CFG["num_workers"],
        pin_memory=True, drop_last=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(
        val_dataset, batch_size=TRAIN_CFG["batch_size"],
        shuffle=False, num_workers=TRAIN_CFG["num_workers"],
        pin_memory=True, persistent_workers=True)

    print(f"  Train: {len(train_dataset)} slices | Val: {len(val_dataset)} slices")
    log_vram(f"pre-Task {task_id}")

    # Optimizer + Scheduler (reset per task)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_CFG["lr"],
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    criterion = CombinedLoss(alpha=TRAIN_CFG["loss_alpha"], ignore_index=IGNORE_INDEX)

    # EWCTrainer uses EWCRegularizer object (not fisher/old_params dicts)
    ewc_trainer = EWCTrainer(
        model=model, optimizer=optimizer, criterion=criterion,
        scheduler=scheduler, device=DEVICE, task_id=task_id,
        checkpoint_dir=CHECKPOINT_DIR, log_dir=LOG_DIR,
        ewc_regularizer=(ewc_reg if USE_EWC_REGULARIZATION else None),
        save_every=TRAIN_CFG["save_every"],
        ignore_index=IGNORE_INDEX,
        cil_eval_every=CIL_EVAL_EVERY,
    )
    ewc_trainer.miou_history = list(global_miou_history)

    # Training — val_cil_loader for mid-training CIL checkpoint selection
    result = ewc_trainer.fit(
        train_loader   = train_loader,
        val_loader     = val_loader,
        n_epochs       = TRAIN_CFG["n_epochs"],
        start_epoch    = 1,
        cil_val_loader = val_cil_loader,   # [VAL] checkpoint selection ONLY; not final report
        task_baselines = task_baselines,
    )
    dice_all, iou_all = result if result is not None else (None, None)

    # Sync mIoU history to disk (resume safety)
    global_miou_history = list(ewc_trainer.miou_history)
    save_miou_history(global_miou_history, MIOU_HISTORY_PATH)
    avg_m = float(np.mean(global_miou_history)) if global_miou_history else float('nan')
    print(f"  mIoU history after task {task_id}: "
          f"{[round(float(x),4) for x in global_miou_history]} | avg={avg_m:.4f}")

    # Update Fisher after task (task-specific mask for current task)
    ewc_state_path = None
    if USE_EWC_REGULARIZATION:
        print(f"\n  Updating Fisher — task_id={task_id} (task-specific mask)...")
        task_fisher_records = [
            r for r in train_records_all
            if set(TASK_ORGANS[task_id]) & set(r["organs_present"])
        ]
        assert_records_split(task_fisher_records, "train")
        fisher_ds_new = BTCVDataset(
            records    = task_fisher_records,
            images_dir = IMAGES_2D_DIR,
            masks_dir  = MASKS_2D_DIR,
            task_id    = task_id,
            augment    = False,
        )
        fisher_loader_new = DataLoader(
            fisher_ds_new, batch_size=FISHER_BATCH_SIZE,
            shuffle=True, num_workers=TRAIN_CFG["num_workers"],
        )
        ewc_reg.compute_and_set_fisher(
            model       = model,
            dataloader  = fisher_loader_new,
            task_organs = TASK_ORGANS[task_id],
            num_samples = FISHER_SAMPLES,
        )

        ewc_state_path = f"{CHECKPOINT_DIR}/ewc_state_after_task{task_id}.pth"
        ewc_reg.save(ewc_state_path, task_id=task_id)
    else:
        print("\n  [ABLATION] Finetune baseline: EWC penalty and Fisher update are OFF.")

    # Update organ baselines for NEW organs only (never overwrite past baselines)
    if dice_all is not None:
        task_baselines = update_organ_baselines(
            task_baselines, dice_all, TASK_ORGANS[task_id], BASELINES_PATH)

    # ── Post-training final evaluation on TEST set ────────────────────────
    # PROTOCOL: After training is complete (fit() has run), we run a clean
    # evaluation on the held-out TEST set. This is separate from the
    # mid-training CIL evaluation, which uses val_cil_loader for checkpoint selection.
    # The test set is NEVER used for model selection (no leakage).
    print(f"\n  ── POST-TRAINING FINAL EVAL on TEST set ──")
    print(f"  TEST loader: {len(test_cil_dataset)} slices (held-out, never used for selection)")
    (dice_all_test, dice_old_test, dice_new_test,
     iou_all_test,  iou_old_test,  iou_new_test) = ewc_trainer.evaluate_cil(test_cil_loader, split="test")

    # Compute forgetting vs baselines (test set)
    fgt_dict_test, mean_fgt_test = ({}, float('nan'))
    if task_baselines:
        fgt_dict_test, mean_fgt_test = ewc_trainer.compute_forgetting(
            dice_all_test, task_baselines)

    iou_vals_test = [v for v in iou_all_test.values() if not (v != v)]  # not NaN
    miou_test = float(np.mean(iou_vals_test)) if iou_vals_test else float('nan')
    dsc_vals_test = [v for v in dice_all_test.values() if not (v != v)]
    mdsc_test = float(np.mean(dsc_vals_test)) if dsc_vals_test else float('nan')
    fgt_str = f"{mean_fgt_test:+.4f}" if not (mean_fgt_test != mean_fgt_test) else "N/A"

    print(f"  [TEST] mDSC={mdsc_test:.4f}  mIoU={miou_test:.4f}  MeanFgt={fgt_str}")

    # Save TEST results to separate JSON (clearly named _test_ to avoid confusion)
    ewc_lam_val = ewc_reg.ewc_lambda if ewc_reg is not None else 0.0
    test_results = {
        "split"         : "TEST",  # EXPLICIT: this is test set, not val
        "method"        : "Online diagonal empirical Fisher EWC-style baseline",
        "task_id"       : task_id,
        "ewc_lambda"    : float(ewc_lam_val),
        "fisher_samples": FISHER_SAMPLES,
        "fisher_batch"  : FISHER_BATCH_SIZE,
        "dsc": {
            "all"      : _safe_float(mdsc_test),
            "per_organ": {ORGAN_NAMES[k]: _safe_float(v)
                          for k, v in dice_all_test.items() if not (v != v)},
        },
        "miou": {
            "all"      : _safe_float(miou_test),
            "per_organ": {ORGAN_NAMES[k]: _safe_float(v)
                          for k, v in iou_all_test.items() if not (v != v)},
        },
        "forgetting": {
            "mean"     : _safe_float(mean_fgt_test),
            "per_organ": {ORGAN_NAMES[k]: _safe_float(v)
                          for k, v in fgt_dict_test.items()},
        },
    }
    test_path = os.path.join(LOG_DIR, f"ewc_task{task_id}_TEST_results.json")
    with open(test_path, 'w') as _f:
        json.dump(test_results, _f, indent=2, ensure_ascii=False)
    print(f"  TEST results saved: {test_path}  [FINAL REPORT — use this for paper]")

    print(f"\n  Task {task_id} DONE!")
    print(f"  EWC state (task-specific Fisher) saved: {ewc_state_path}") if ewc_state_path else print("  EWC state: not saved for finetune ablation") if ewc_state_path else print("  EWC state: not saved for finetune ablation") if ewc_state_path else print("  EWC state: not saved for finetune ablation") if ewc_state_path else print("  EWC state: not saved for finetune ablation") if ewc_state_path else print("  EWC state: not saved for finetune ablation") if ewc_state_path else print("  EWC state: not saved for finetune ablation")
    print(f"  Organ baselines: {len(task_baselines)} organs tracked.")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n\n{'='*70}")
print(f"  ALL TASKS COMPLETED — Online diagonal empirical Fisher EWC-style baseline")
print(f"  Evaluation reports: {LOG_DIR}/ewc_task*_cil_results.json")
print(f"{'='*70}")







## Step 6 — Tong Hop Ket Qua

In [ ]:
# ============================================================
# Tong hop ket qua EWC qua tat ca tasks
# Doc schema JSON moi: {dsc, miou, forgetting}
# ============================================================

print("="*75)
print("  EWC BASELINE - TONG HOP KET QUA")
print("="*75)

all_cil_results = {}

for task_id in range(1, 5):
    cil_path = f"{LOG_DIR}/ewc_task{task_id}_cil_results.json"

    if os.path.exists(cil_path):
        with open(cil_path, 'r') as f:
            results = json.load(f)
        all_cil_results[task_id] = results

        # --- In tong quan theo schema moi ---
        print(f"\n  Task {task_id}:")
        dsc_block  = results.get('dsc',  {})
        miou_block = results.get('miou', {})
        fgt_block  = results.get('forgetting', {})

        print(f"    DSC  : Old={dsc_block.get('old','N/A')}  "
              f"New={dsc_block.get('new','N/A')}  "
              f"All={dsc_block.get('all','N/A')}")
        print(f"    mIoU : C1={miou_block.get('c1','N/A')}  "
              f"C2:T={miou_block.get('c2t','N/A')}  "
              f"All={miou_block.get('all','N/A')}  "
              f"avg={miou_block.get('avg','N/A')}")
        mean_fgt = fgt_block.get('mean', None)
        if mean_fgt is not None:
            print(f"    Fgt  : mean={mean_fgt:+.4f}")

        # Per-organ DSC
        per_organ_dsc = dsc_block.get('per_organ', {})
        for organ_name, dice_val in per_organ_dsc.items():
            print(f"      {organ_name:30s}: DSC={dice_val}")
    else:
        print(f"\n  Task {task_id}: File not found ({cil_path})")

# --- Progression table DSC + mIoU ---
print(f"\n{'='*75}")
print("  DSC & mIoU PROGRESSION QUA CAC TASKS:")
print(f"{'='*75}")
all_organ_ids = sorted(ALL_PAST_ORGANS[4])
header = f"  {'Organ':30s}" + "".join(f" Task{t} DSC  IoU " for t in range(1, 5))
print(header)
print("  " + "-" * (30 + 18 * 4))
for oid in all_organ_ids:
    name = ORGAN_NAMES[oid].split('(')[0].strip()[:28]
    row  = f"  {name:30s}"
    for tid in range(1, 5):
        if tid in all_cil_results:
            organ_name_full = ORGAN_NAMES[oid]
            dsc = all_cil_results[tid].get('dsc',{}).get('per_organ',{}).get(organ_name_full, None)
            iou = all_cil_results[tid].get('miou',{}).get('per_organ',{}).get(organ_name_full, None)
            dsc_s = f"{dsc:.4f}" if dsc is not None else "  -   "
            iou_s = f"{iou:.4f}" if iou is not None else "  -   "
            row += f" {dsc_s} {iou_s}"
        else:
            row += f"    ?       ?  "
    print(row)

# --- Summary metrics cuoi cung (Task 4) ---
print(f"\n{'='*75}")
print("  SUMMARY METRICS SAU TASK 4 (CIL CUOI):")
print(f"{'='*75}")
if 4 in all_cil_results:
    r4 = all_cil_results[4]
    d4 = r4.get('dsc',  {})
    m4 = r4.get('miou', {})
    print(f"  DSC Old  (C1=Liver+Stomach)     : {d4.get('old','N/A')}")
    print(f"  DSC New  (C2:T organs moi)      : {d4.get('new','N/A')}")
    print(f"  DSC All  (toan bo 13 organs)    : {d4.get('all','N/A')}")
    print(f"  mIoU C1  (rigidity, chong quen) : {m4.get('c1','N/A')}")
    print(f"  mIoU C2:T (plasticity, hoc moi) : {m4.get('c2t','N/A')}")
    print(f"  mIoU All  (tong the)            : {m4.get('all','N/A')}")
    print(f"  avg mIoU  (tich hop toan qua trinh CIL): {m4.get('avg','N/A')}")
    fgt4 = r4.get('forgetting', {}).get('mean', None)
    if fgt4 is not None:
        print(f"  Mean Forgetting                 : {fgt4:+.4f}")

# --- So sanh voi Fine-tuning (neu co) ---
print(f"\n{'='*75}")
print("  So sanh EWC vs Fine-tuning (sau Task 4):")
print(f"{'='*75}")
ft_path  = f"{LOG_DIR}/task4_cil_results.json"
ewc_path = f"{LOG_DIR}/ewc_task4_cil_results.json"

if os.path.exists(ft_path):
    with open(ft_path, 'r') as f:
        ft_r = json.load(f)
    # Ho tro ca schema cu (overall_dice) lan schema moi (dsc.all)
    ft_dsc = ft_r.get('dsc',{}).get('all') or ft_r.get('overall_dice')
    ft_fgt = ft_r.get('forgetting',{}).get('mean') or ft_r.get('mean_forgetting')
    ft_miou = ft_r.get('miou',{}).get('all', 'N/A')
    print(f"  Fine-tuning : DSC All={ft_dsc}  mIoU All={ft_miou}  Fgt={ft_fgt}")

if os.path.exists(ewc_path):
    with open(ewc_path, 'r') as f:
        ewc_r = json.load(f)
    ewc_dsc  = ewc_r.get('dsc',{}).get('all')
    ewc_fgt  = ewc_r.get('forgetting',{}).get('mean')
    ewc_miou = ewc_r.get('miou',{}).get('all')
    ewc_avg  = ewc_r.get('miou',{}).get('avg')
    print(f"  EWC         : DSC All={ewc_dsc}  mIoU All={ewc_miou}  "
          f"avg mIoU={ewc_avg}  Fgt={ewc_fgt}")

print(f"\n  --> Ket qua nay se dung de so sanh trong paper!")
print(f"  --> avg mIoU phan anh hieu nang TOAN BO qua trinh CIL, khong chi task cuoi.")


## Step 7 — Performance Matrix R[i][j]

**Performance Matrix** $R \in \mathbb{R}^{T \times T}$ là công cụ chuẩn trong CIL evaluation:

$$R[i][j] = \text{DSC (hoặc mIoU) trên Task } j \text{ sau khi đã train xong Task } i$$

- **Đường chéo** $R[i][i]$: plasticity — khả năng học task mới
- **Tam giác dưới** $R[i][j], j < i$: rigidity — giữ kiến thức cũ
- **Tam giác trên**: không định nghĩa (chưa học task $j$ tại thời điểm $i$)

### Derived Metrics từ R:
- **Average Accuracy**: $\bar{A}_T = \frac{1}{T} \sum_{j=1}^{T} R[T][j]$
- **Forgetting per task**: $F_j = \max_{i \in \{1,...,T-1\}} R[i][j] - R[T][j]$
- **Backward Transfer**: $BWT = \frac{1}{T-1} \sum_{j=1}^{T-1} (R[T][j] - R[j][j])$

In [ ]:
# ============================================================
# CLEAN-RUN SAFETY: no in-place checkpoint patching
# ============================================================
# Historical versions of this notebook patched old checkpoints after training.
# That is not reviewer-friendly and violates clean-run reproducibility. This
# cell now audits required state files and fails fast if anything is missing.

print("=== EWC state audit (no checkpoint patching) ===")
for task_id in range(1, 5):
    ewc_state_path = f"{CHECKPOINT_DIR}/ewc_state_after_task{task_id}.pth"
    if not os.path.exists(ewc_state_path):
        raise FileNotFoundError(
            f"Missing EWC state for Task {task_id}: {ewc_state_path}. "
            "Run the training cells from a clean kernel instead of patching checkpoints."
        )
    state = torch.load(ewc_state_path, map_location="cpu", weights_only=False)
    fisher = state.get("fisher") or {}
    old_params = state.get("old_params") or state.get("optpar") or {}
    assert fisher, f"Missing Fisher tensors in {ewc_state_path}"
    assert old_params, f"Missing old_params/optpar tensors in {ewc_state_path}"
    vals = torch.cat([v.float().reshape(-1) for v in fisher.values()])
    assert torch.isfinite(vals).all(), f"NaN/Inf Fisher in {ewc_state_path}"
    assert float(vals.sum()) > 0.0, f"Fisher collapse in {ewc_state_path}"
    print(f"  Task {task_id}: OK | fisher_mean={float(vals.mean()):.3e} "
          f"max={float(vals.max()):.3e} min={float(vals.min()):.3e}")

print("[ASSERTION PASSED] Required EWC states exist and are finite. No patching performed.")



In [ ]:
# ============================================================
# Performance Matrix R[i][j] — TEST SET final evaluation
# ============================================================
# R[i][j] = mean DSC on Task j organs after training Task i.
# Final paper-reportable metrics are computed on TEST only.
# ============================================================

def force_cleanup():
    """Release VRAM after each checkpoint evaluation."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _finite_metric(x, name):
    x = float(x)
    assert np.isfinite(x), f"Invalid metric {name}: {x}"
    return x


def resolve_task_checkpoint(checkpoint_dir, task_id):
    candidates = []
    if task_id == 1:
        candidates.append(f"{checkpoint_dir}/task1_best.pth")
        candidates.append(f"{DRIVE_ROOT}/checkpoints_baseline/task1_best.pth")
    else:
        candidates.extend([
            f"{checkpoint_dir}/ewc_task{task_id}_best_cil.pth",
            f"{checkpoint_dir}/ewc_task{task_id}_best.pth",
        ])
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Missing checkpoint for Task {task_id}; checked {candidates}")


def compute_performance_matrix(checkpoint_dir, test_records, images_dir, masks_dir,
                               device, task_organ_map, organ_names, split="test",
                               num_tasks=4, num_classes=14, encoder="resnet34",
                               ignore_index=255, batch_size=16, num_workers=2):
    """Compute CIL performance matrix on a declared split; final use requires split='test'."""
    assert split == "test", "Final performance matrix must use split == 'test'"
    assert_records_split(test_records, "test")
    assert len(test_records) > 0, "No TEST records available for final evaluation"

    test_dataset_full = BTCVDataset(
        records    = test_records,
        images_dir = images_dir,
        masks_dir  = masks_dir,
        task_id    = None,
        augment    = False,
    )
    test_loader = DataLoader(
        test_dataset_full, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    print(f"  [TEST] R-matrix loader: {len(test_dataset_full)} slices (full mask, task_id=None)")

    R_dsc = np.full((num_tasks, num_tasks), np.nan)
    R_iou = np.full((num_tasks, num_tasks), np.nan)
    per_organ_dsc, per_organ_iou = {}, {}

    for i in range(1, num_tasks + 1):
        ckpt_path = resolve_task_checkpoint(checkpoint_dir, i)
        model_i = build_unet(num_classes=num_classes, encoder=encoder,
                             pretrained=None).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        assert "model_state" in ckpt, f"Invalid checkpoint missing model_state: {ckpt_path}"
        model_i.load_state_dict(ckpt["model_state"])
        model_i.eval()
        print(f"  [TEST] Evaluating model after Task {i} "
              f"(epoch {ckpt.get('epoch', 'NA')}, {os.path.basename(ckpt_path)})")

        all_organs_up_to_i = []
        for t in range(1, i + 1):
            all_organs_up_to_i.extend(task_organ_map[t])

        inter, denom = defaultdict(float), defaultdict(float)
        with torch.no_grad():
            for ct_batch, mask_batch in test_loader:
                ct_batch = ct_batch.to(device, non_blocking=True)
                mask_batch = mask_batch.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    logits = model_i(ct_batch)
                pred = torch.argmax(logits, dim=1)
                valid = (mask_batch != ignore_index)
                for oid in all_organs_up_to_i:
                    pred_o = (pred == oid) & valid
                    true_o = (mask_batch == oid) & valid
                    batch_inter = (pred_o & true_o).sum().item()
                    inter[oid] += batch_inter
                    denom[oid] += pred_o.sum().item() + true_o.sum().item()

        organ_dsc, organ_iou = {}, {}
        for oid in all_organs_up_to_i:
            organ_dsc[oid] = 2.0 * inter[oid] / denom[oid] if denom[oid] > 0 else float('nan')
            union_iou = denom[oid] - inter[oid]
            organ_iou[oid] = inter[oid] / union_iou if union_iou > 0 else float('nan')

        for j in range(1, i + 1):
            task_j_organs = task_organ_map[j]
            dsc_vals = [organ_dsc[o] for o in task_j_organs if o in organ_dsc and not np.isnan(organ_dsc[o])]
            iou_vals = [organ_iou[o] for o in task_j_organs if o in organ_iou and not np.isnan(organ_iou[o])]
            R_dsc[i-1, j-1] = _finite_metric(np.mean(dsc_vals), f"R_dsc[{i},{j}]") if dsc_vals else np.nan
            R_iou[i-1, j-1] = _finite_metric(np.mean(iou_vals), f"R_iou[{i},{j}]") if iou_vals else np.nan
            per_organ_dsc[(i, j)] = {o: organ_dsc.get(o, np.nan) for o in task_j_organs}
            per_organ_iou[(i, j)] = {o: organ_iou.get(o, np.nan) for o in task_j_organs}

        del model_i, ckpt
        force_cleanup()

    return R_dsc, R_iou, per_organ_dsc, per_organ_iou


def print_performance_matrix(R, metric_name="DSC"):
    T = R.shape[0]
    print(f"\n{'='*70}")
    print(f"  [TEST] PERFORMANCE MATRIX R[i][j] — {metric_name}")
    print(f"{'='*70}")
    header = f"  {'After/On':>12s}" + ''.join(f"  Task {j:d}  " for j in range(1, T+1)) + "  | Mean"
    print(header)
    print("  " + "-" * (14 + 10*T + 8))
    row_means = []
    for i in range(T):
        vals, row = [], f"  {'Task '+str(i+1):>12s}"
        for j in range(T):
            val = R[i, j]
            if np.isnan(val):
                row += "    -    "
            else:
                row += f"  {val:.4f} "
                vals.append(val)
        row_mean = float(np.mean(vals)) if vals else np.nan
        row_means.append(row_mean)
        row += f"  | {row_mean:.4f}" if not np.isnan(row_mean) else "  |   -  "
        print(row)
    return row_means


def compute_cil_metrics_from_R(R_dsc, R_iou):
    T = R_dsc.shape[0]
    last_row_dsc = [R_dsc[T-1, j] for j in range(T) if not np.isnan(R_dsc[T-1, j])]
    last_row_iou = [R_iou[T-1, j] for j in range(T) if not np.isnan(R_iou[T-1, j])]
    fgt_list = []
    for j in range(T - 1):
        col_vals = [R_dsc[i, j] for i in range(T) if not np.isnan(R_dsc[i, j])]
        if len(col_vals) >= 2:
            fgt_list.append(max(col_vals[:-1]) - col_vals[-1])
    bwt_list = [R_dsc[T-1, j] - R_dsc[j, j] for j in range(T - 1)
                if not np.isnan(R_dsc[T-1, j]) and not np.isnan(R_dsc[j, j])]
    row_means = [np.mean([R_dsc[i, j] for j in range(T) if not np.isnan(R_dsc[i, j])])
                 for i in range(T) if any(not np.isnan(R_dsc[i, j]) for j in range(T))]
    return {
        "avg_accuracy_dsc": _finite_metric(np.mean(last_row_dsc), "avg_accuracy_dsc") if last_row_dsc else float('nan'),
        "avg_accuracy_iou": _finite_metric(np.mean(last_row_iou), "avg_accuracy_iou") if last_row_iou else float('nan'),
        "mean_forgetting_dsc": _finite_metric(np.mean(fgt_list), "mean_forgetting_dsc") if fgt_list else float('nan'),
        "backward_transfer_dsc": _finite_metric(np.mean(bwt_list), "backward_transfer_dsc") if bwt_list else float('nan'),
        "avg_incremental_accuracy": _finite_metric(np.mean(row_means), "avg_incremental_accuracy") if row_means else float('nan'),
    }


def save_matrix_plots(R_dsc, R_iou, metrics, log_dir, prefix="ewc"):
    os.makedirs(log_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, R, title in zip(axes, [R_dsc, R_iou], ["DSC", "mIoU"]):
        im = ax.imshow(R, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
        ax.set_xticks(range(R.shape[1])); ax.set_yticks(range(R.shape[0]))
        ax.set_xticklabels([f"Task {j+1}" for j in range(R.shape[1])])
        ax.set_yticklabels([f"After Task {i+1}" for i in range(R.shape[0])])
        ax.set_xlabel("Evaluated on TEST organs")
        ax.set_ylabel("Model state")
        ax.set_title(f"TEST performance matrix — {title}")
        for i in range(R.shape[0]):
            for j in range(R.shape[1]):
                val = R[i, j]
                ax.text(j, i, "-" if np.isnan(val) else f"{val:.3f}",
                        ha='center', va='center', fontsize=10,
                        color='white' if not np.isnan(val) and val < 0.5 else 'black')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    heatmap_path = f"{log_dir}/{prefix}_performance_matrix_test.png"
    plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
    plt.show()

    tasks = np.arange(1, R_dsc.shape[0] + 1)
    dice_over_task = [np.nanmean(R_dsc[i, :i+1]) for i in range(R_dsc.shape[0])]
    forgetting_curve = []
    for i in range(R_dsc.shape[0]):
        vals = []
        for j in range(i):
            col = [R_dsc[k, j] for k in range(i + 1) if not np.isnan(R_dsc[k, j])]
            if len(col) >= 2:
                vals.append(max(col[:-1]) - col[-1])
        forgetting_curve.append(float(np.mean(vals)) if vals else 0.0)

    plt.figure(figsize=(7, 4))
    plt.plot(tasks, dice_over_task, marker='o')
    plt.ylim(0, 1); plt.xticks(tasks)
    plt.xlabel("After task"); plt.ylabel("Mean TEST Dice")
    plt.title("Dice over task")
    plt.grid(True, alpha=0.3)
    dice_path = f"{log_dir}/{prefix}_dice_over_task_test.png"
    plt.savefig(dice_path, dpi=150, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(tasks, forgetting_curve, marker='o', color='tab:red')
    plt.xticks(tasks)
    plt.xlabel("After task"); plt.ylabel("Mean forgetting (DSC)")
    plt.title("Forgetting curve")
    plt.grid(True, alpha=0.3)
    fgt_path = f"{log_dir}/{prefix}_forgetting_curve_test.png"
    plt.savefig(fgt_path, dpi=150, bbox_inches='tight')
    plt.show()
    return {"heatmap": heatmap_path, "dice_over_task": dice_path, "forgetting_curve": fgt_path}


print("\n" + "#"*70)
print("#  [TEST] PERFORMANCE MATRIX — Online EWC-style baseline")
print("#  Validation was used only for checkpoint selection.")
print("#"*70)
assert_records_split(test_records_all, "test")
assert not any(r["volume_id"] in val_vol_ids for r in test_records_all), "Validation leakage into final metrics"
print("[ASSERTION PASSED] no validation data used in final metrics")

R_dsc, R_iou, per_organ_dsc, per_organ_iou = compute_performance_matrix(
    checkpoint_dir = CHECKPOINT_DIR,
    test_records   = test_records_all,
    images_dir     = IMAGES_2D_DIR,
    masks_dir      = MASKS_2D_DIR,
    device         = DEVICE,
    task_organ_map = TASK_ORGANS,
    organ_names    = ORGAN_NAMES,
    split          = "test",
    num_tasks      = 4,
    num_classes    = TRAIN_CFG["num_classes"],
    encoder        = TRAIN_CFG["encoder"],
    ignore_index   = IGNORE_INDEX,
    batch_size     = TRAIN_CFG["batch_size"],
    num_workers    = TRAIN_CFG["num_workers"],
)

print_performance_matrix(R_dsc, metric_name="DSC")
print_performance_matrix(R_iou, metric_name="mIoU")
cil_metrics = compute_cil_metrics_from_R(R_dsc, R_iou)
print(f"\n[TEST] Avg Dice={cil_metrics['avg_accuracy_dsc']:.4f} | "
      f"Forgetting={cil_metrics['mean_forgetting_dsc']:+.4f}")

perf_matrix_results = {
    "method": "Online diagonal empirical Fisher EWC-style baseline",
    "split": "test",
    "assert_no_validation_leakage": True,
    "seed": int(CURRENT_SEED),
    "ablation": CURRENT_ABLATION,
    "R_dsc": R_dsc.tolist(),
    "R_iou": R_iou.tolist(),
    "metrics": cil_metrics,
    "per_organ_dsc": {f"R[{i}][{j}]": {ORGAN_NAMES[oid]: round(float(v), 4)
                       for oid, v in organs.items() if not np.isnan(v)}
                       for (i, j), organs in per_organ_dsc.items()},
    "per_organ_iou": {f"R[{i}][{j}]": {ORGAN_NAMES[oid]: round(float(v), 4)
                       for oid, v in organs.items() if not np.isnan(v)}
                       for (i, j), organs in per_organ_iou.items()},
}
perf_matrix_path = f"{LOG_DIR}/ewc_performance_matrix_test.json"
with open(perf_matrix_path, 'w') as f:
    json.dump(perf_matrix_results, f, indent=2, default=str)
plot_paths = save_matrix_plots(R_dsc, R_iou, cil_metrics, LOG_DIR, prefix="ewc")
print(f"[TEST] Performance matrix saved: {perf_matrix_path}")
print(f"[TEST] Plots saved: {plot_paths}")



In [ ]:
# ============================================================
# Multi-seed / ablation aggregation and reviewer summary export
# ============================================================
# Aggregates completed runs for seeds 42, 3407, 1234 and ablations:
# Finetune, EWC lambda=100, EWC lambda=500. Missing runs are reported rather
# than silently imputed.

def _run_log_dir(seed, ablation_name):
    return f"{DRIVE_ROOT}/logs_ewc_v2/{ablation_name}/seed_{seed}"


def collect_completed_run_summaries(base_log_dir=LOG_DIR):
    rows = []
    search_roots = [base_log_dir]
    for ab in ABLATION_GRID:
        for seed in EXPERIMENT_SEEDS:
            search_roots.append(_run_log_dir(seed, ab["name"]))
    seen = set()
    for root in search_roots:
        path = os.path.join(root, "ewc_performance_matrix_test.json")
        if path in seen or not os.path.exists(path):
            continue
        seen.add(path)
        with open(path, 'r') as f:
            r = json.load(f)
        assert r.get("split", "").lower() == "test", f"Non-test result in final aggregation: {path}"
        m = r.get("metrics", {})
        ablation = r.get("ablation", {}) or {}
        rows.append({
            "path": path,
            "seed": int(r.get("seed", -1)),
            "ablation": ablation.get("name", "current_run"),
            "ewc_lambda": float(ablation.get("ewc_lambda", r.get("ewc_lambda", float('nan')))),
            "avg_dice": float(m.get("avg_accuracy_dsc", float('nan'))),
            "forgetting": float(m.get("mean_forgetting_dsc", float('nan'))),
            "avg_iou": float(m.get("avg_accuracy_iou", float('nan'))),
        })
    return rows


def summarize_mean_std(rows):
    grouped = defaultdict(list)
    for r in rows:
        assert np.isfinite(r["avg_dice"]), f"Invalid avg_dice in {r['path']}"
        assert np.isfinite(r["forgetting"]), f"Invalid forgetting in {r['path']}"
        grouped[r["ablation"]].append(r)
    summary = []
    for ablation, items in sorted(grouped.items()):
        dice = np.array([x["avg_dice"] for x in items], dtype=float)
        fgt = np.array([x["forgetting"] for x in items], dtype=float)
        summary.append({
            "ablation": ablation,
            "n_seeds": int(len(items)),
            "seeds": sorted(int(x["seed"]) for x in items),
            "avg_dice_mean": float(np.mean(dice)),
            "avg_dice_std": float(np.std(dice, ddof=1)) if len(dice) > 1 else 0.0,
            "forgetting_mean": float(np.mean(fgt)),
            "forgetting_std": float(np.std(fgt, ddof=1)) if len(fgt) > 1 else 0.0,
        })
    return summary


completed_rows = collect_completed_run_summaries(LOG_DIR)
summary_rows = summarize_mean_std(completed_rows)

summary_json = f"{LOG_DIR}/final_experiment_summary.json"
summary_csv = f"{LOG_DIR}/final_experiment_summary.csv"
comparison_csv = f"{LOG_DIR}/ablation_comparison_table.csv"

with open(summary_json, 'w') as f:
    json.dump({
        "method_scope": "Online diagonal empirical Fisher EWC-style regularization baseline",
        "final_metrics_split": "test",
        "validation_usage": "checkpoint selection and tuning only",
        "expected_seeds": EXPERIMENT_SEEDS,
        "expected_ablations": ABLATION_GRID,
        "completed_runs": completed_rows,
        "summary": summary_rows,
    }, f, indent=2)

with open(summary_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["ablation", "n_seeds", "seeds", "avg_dice_mean", "avg_dice_std", "forgetting_mean", "forgetting_std"])
    writer.writeheader()
    writer.writerows(summary_rows)

with open(comparison_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["ablation", "seed", "ewc_lambda", "avg_dice", "forgetting", "avg_iou", "path"])
    writer.writeheader()
    writer.writerows(completed_rows)

print("\nFINAL EXPERIMENT SUMMARY (TEST SET ONLY)")
print("ablation        n  Avg Dice mean±std    Forgetting mean±std")
for r in summary_rows:
    print(f"{r['ablation']:14s} {r['n_seeds']:d}  "
          f"{r['avg_dice_mean']:.4f} ± {r['avg_dice_std']:.4f}    "
          f"{r['forgetting_mean']:+.4f} ± {r['forgetting_std']:.4f}")
missing = []
for ab in ABLATION_GRID:
    got = {r["seed"] for r in completed_rows if r["ablation"] == ab["name"]}
    for seed in EXPERIMENT_SEEDS:
        if seed not in got:
            missing.append((ab["name"], seed))
if missing:
    print(f"\n[MISSING RUNS] {missing}")
    print("Summary is exportable but not complete for all requested seeds/ablations yet.")
print(f"\nSaved: {summary_json}")
print(f"Saved: {summary_csv}")
print(f"Saved: {comparison_csv}")

